# CBAM-DenseNet121 — Final Reviewer-Response Pipeline (v2)

This is the corrected version incorporating a second review pass. Changes from v1:

| # | Change | Why |
|---|---|---|
| 1 | **4-model ablation**: DenseNet121, Channel-Attention-only, Spatial-Attention-only, CBAM (channel+spatial) | Isolates the individual contribution of each attention component, not just the combined effect |
| 2 | `EPOCHS = 40` with early stopping (was 15) | 15 matched only the old Phase I run; the Phase II ablation gets a real convergence budget |
| 3 | Checkpoint now monitors `val_loss` / `mode="min"` (was `val_accuracy` / `max`) | Consistent with early stopping and LR scheduling, which already use `val_loss` |
| 4 | Phase II test set evaluates a **pre-specified** `FINAL_SEED = 42` for every model (was "best validation seed") | Selecting the best-of-5 seed before the test evaluation is itself a data-driven choice — using a pre-registered seed removes that degree of freedom |
| 5 | Near-duplicate grouping is now **global across all classes** (was per-class only) | A per-class-only comparison can never detect a cross-class near-duplicate, which would be a real, more serious leakage/labeling problem; this version explicitly audits and reports cross-class groups |
| 6 | External dataset changed to `shreyanmohanty/oasis-alzheimers-detection-multi-class-dataset` (was `ninadaithal/imagesoasis`) | The new dataset documents subject-level structure (416 subjects), enabling subject-level (not just slice-level) external evaluation |
| 7 | External evaluation aggregates predictions **per subject** when a subject ID can be parsed from filenames, with an explicit fallback and warning if it cannot | Slice-level accuracy on a multi-slice-per-subject external set overstates independence unless subjects are the unit of evaluation |
| 8 | Statistical significance now reports **95% confidence intervals** (t-distribution, n=5) alongside Wilcoxon/paired-t/Cohen's d | A p-value and effect size alone under-communicate the uncertainty from only 5 seeds |
| 9 | Confusion matrices and one-vs-rest ROC curves are generated **for every model** in the Phase II locked test evaluation, not just the proposed model | Needed to visually support the ablation claims, not just report scalar metrics |

**Before running (local / JupyterLab):** download the two Kaggle datasets below and place them
under a local `data/` folder (or set the `DATA_ROOT` environment variable to wherever you keep them):
- `uraninjo/augmented-alzheimer-mri-dataset-v2` → `data/augmented-alzheimer-mri-dataset-v2/` (training dataset)
- `shreyanmohanty/oasis-alzheimers-detection-multi-class-dataset` → `data/oasis-alzheimers-detection-multi-class-dataset/` (external validation dataset)

You can grab them with the Kaggle CLI, e.g.:
```bash
pip install kaggle
kaggle datasets download -d uraninjo/augmented-alzheimer-mri-dataset-v2 -p data/augmented-alzheimer-mri-dataset-v2 --unzip
kaggle datasets download -d shreyanmohanty/oasis-alzheimers-detection-multi-class-dataset -p data/oasis-alzheimers-detection-multi-class-dataset --unzip
```

Then **Run All** in JupyterLab. This version trains **4 models × 5 seeds = 20 runs** in Block 3, so
expect roughly 2–3× the runtime of v1 — budget accordingly for your local/remote GPU. Block 3 is
resumable (skips any (model, seed) pair already recorded), so it is safe to stop and restart across
sessions.


In [ ]:
!pip install scikit-learn tensorflow pandas numpy matplotlib seaborn scipy pillow imagehash

In [ ]:
import tensorflow as tf
print(tf.__version__)
print(tf.config.list_physical_devices('GPU'))

In [ ]:
# ============================================================
# BLOCK 0 — IMPORTS, SEEDS, CONFIGURATION
# ============================================================
import os, gc, json, time, random, hashlib, re, subprocess, sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from tensorflow.keras import layers, Model, backend as K
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input as densenet_preprocess
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix,
                             classification_report, roc_auc_score, roc_curve, auc)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import label_binarize
from scipy import stats

for pkg in ["imagehash"]:
    try: __import__(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
import imagehash
from PIL import Image

# ---------------- task ----------------
CLASS_NAMES = ["NonDemented", "VeryMildDemented", "MildDemented", "ModerateDemented"]
NUM_CLASSES = len(CLASS_NAMES)
IMG_SIZE, BATCH_SIZE = 224, 32
SEEDS, FINAL_SEED = [13, 27, 42, 77, 101], 42
TARGET_SPLIT = {"train": 0.70, "val": 0.15, "test": 0.15}

# ---------------- architecture (Block 2) ----------------
ATTENTION_INIT  = "zeros"   # "he_normal" reproduces the old, handicapped behaviour
SPATIAL_KERNEL  = 3         # a 7x7 kernel on a 7x7 map is global, not spatial
CHANNEL_RATIO   = 8
MULTI_SCALE     = True      # attention at 14x14 as well as 7x7
UNFREEZE_LAST_N = 120       # was 30; attention needs adaptable features

# ---------------- training (Block 3) ----------------
STAGE1_EPOCHS, STAGE2_EPOCHS = 8, 40
STAGE1_LR, STAGE2_LR = 1e-3, 3e-5
LABEL_SMOOTHING, ES_PATIENCE = 0.05, 10
USE_TTA = True

# ---------------- paths ----------------
DATA_ROOT = Path(os.environ.get("DATA_ROOT", str(Path.home() / "Downloads")))
RAW_DATASET_CANDIDATES = [DATA_ROOT / "rk kaggle mri v2"]

WORK_DIR = Path("./working")
GOV_DIR      = WORK_DIR / "phase2_governance"; GOV_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR  = WORK_DIR / "results";           RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR  = RESULTS_DIR / "figures";        FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def set_global_seed(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed); np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)


set_global_seed(FINAL_SEED)


def find_raw_dataset_root() -> Path:
    for candidate in RAW_DATASET_CANDIDATES:
        root = Path(candidate)
        if not root.exists():
            continue
        for sub in [root, root / "data"]:
            if sub.exists() and all((sub / s / c).exists()
                                    for s in ["train", "val"] for c in CLASS_NAMES):
                return sub
    for p in DATA_ROOT.rglob("*"):
        if p.is_dir() and p.name in ("train", "val"):
            if all((p / c).exists() for c in CLASS_NAMES) and \
               all((p.parent / s).exists() for s in ["train", "val"]):
                return p.parent
    raise FileNotFoundError(
        f"Could not locate the dataset. Checked {RAW_DATASET_CANDIDATES} and searched "
        f"under {DATA_ROOT}. Need <root>/train/<class> and <root>/val/<class>.")


RAW_ROOT = find_raw_dataset_root()
for g in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(g, True)

print("Raw dataset root :", RAW_ROOT)
print("TensorFlow       :", tf.__version__)
print("GPUs             :", tf.config.list_physical_devices("GPU"))
print(f"Attention        : init={ATTENTION_INIT} k={SPATIAL_KERNEL} "
      f"multi_scale={MULTI_SCALE} unfreeze={UNFREEZE_LAST_N}")
print(f"Training         : stage1={STAGE1_EPOCHS}@{STAGE1_LR} "
      f"stage2={STAGE2_EPOCHS}@{STAGE2_LR} smooth={LABEL_SMOOTHING} tta={USE_TTA}")
print(f"Seeds            : {SEEDS}  final={FINAL_SEED}")


In [ ]:
# ============================================================
# BLOCK 1 — GOVERNANCE VIA ANCHOR-BASED PROVENANCE GROUPING
# ============================================================
# Your previous run collapsed: transitive closure over near-duplicate pairs gave
# one component covering 99.98% of the pool at Hamming 20, still 31.88% at 2.
# Every axial brain slice resembles every other, so similarity chains until one
# group swallows the corpus -- and a group holding a third of the data cannot be
# allocated to one partition without wrecking the ratios (yours came out
# NonDemented 52/25/24 against a 70/15/15 target).
#
# The repository ships originals (val/, 6,400) and augmented derivatives
# (train/, 33,984). Provenance is therefore KNOWN, not inferred: each derivative
# is assigned to its nearest original of the SAME class, and that original is the
# group. No chaining is possible and group size is bounded.

HASH_SIZE = 16              # 16x16 = 256-bit average hash
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}


def scan_dir(root: Path) -> pd.DataFrame:
    rows = []
    for c in CLASS_NAMES:
        d = root / c
        if not d.exists():
            continue
        for p in d.rglob("*"):
            if p.is_file() and p.suffix.lower() in IMG_EXT:
                rows.append({"filepath": str(p), "class": c})
    return pd.DataFrame(rows)


def sha256_of_file(path: str) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 16), b""):
            h.update(chunk)
    return h.hexdigest()


def _bits_of(path: str, hash_size=HASH_SIZE):
    with Image.open(path) as im:
        h = imagehash.average_hash(im.convert("L"), hash_size=hash_size)
    return h.hash.flatten().astype(np.uint8)


def _bits_matrix(paths, hash_size=HASH_SIZE, label=""):
    n, nb = len(paths), hash_size * hash_size
    M = np.zeros((n, nb), dtype=np.uint8)
    ok = np.ones(n, dtype=bool)
    for i, p in enumerate(paths):
        try:
            M[i] = _bits_of(p, hash_size)
        except Exception:
            ok[i] = False
        if i % 8000 == 0:
            print(f"    hashed {label} {i:,}/{n:,}")
    return M, ok


manifest_path = GOV_DIR / "phase2_manifest.csv"
if manifest_path.exists():
    print(f"Manifest exists at {manifest_path}. DELETE IT to rebuild with anchored grouping.")
    manifest = pd.read_csv(manifest_path)
else:
    anchors = scan_dir(RAW_ROOT / "val").assign(role="original")
    derivs  = scan_dir(RAW_ROOT / "train").assign(role="augmented")
    print(f"originals(anchors)={len(anchors):,}   augmented(derivatives)={len(derivs):,}")

    both = pd.concat([anchors, derivs], ignore_index=True)
    print("Hashing with SHA-256 ...")
    both["sha256"] = [sha256_of_file(p) for p in both["filepath"]]
    n0 = len(both)
    both = both.drop_duplicates(subset="sha256", keep="first").reset_index(drop=True)
    print("=" * 70)
    print("Table 3 — SHA-256 exact-duplicate audit")
    print("=" * 70)
    print(f"Input images            : {n0:,}")
    print(f"Unique SHA-256 hashes   : {len(both):,}")
    print(f"Duplicate copies removed: {n0 - len(both):,}")
    print(f"Deduplication rate      : {(n0-len(both))/n0*100:.3f}%")

    A = both[both.role == "original"].reset_index(drop=True)
    D = both[both.role == "augmented"].reset_index(drop=True)
    print(f"\nAnchor grouping: {len(D):,} derivatives -> {len(A):,} anchors")
    Ab, Aok = _bits_matrix(A["filepath"].tolist(), label="anchors")
    Db, Dok = _bits_matrix(D["filepath"].tolist(), label="derivatives")

    Acls = A["class"].to_numpy()
    Ai16 = Ab.astype(np.int16)
    gid  = np.full(len(D), -1, dtype=np.int64)
    dist = np.full(len(D), -1, dtype=np.int32)
    Asum = Ai16.sum(1)
    BLOCK, NB = 2048, HASH_SIZE * HASH_SIZE

    for s in range(0, len(D), BLOCK):
        e = min(s + BLOCK, len(D))
        idx = [i for i in range(s, e) if Dok[i]]
        if not idx:
            continue
        Bi = Db[idx].astype(np.int16)
        H = Bi.sum(1)[:, None] + Asum[None, :] - 2 * (Bi @ Ai16.T)   # Hamming
        dcls = D["class"].to_numpy()[idx]
        H = np.where(dcls[:, None] != Acls[None, :], np.int16(NB + 1), H)
        best = H.argmin(1)
        for k, i in enumerate(idx):
            gid[i], dist[i] = best[k], int(H[k, best[k]])
        if s % (BLOCK * 8) == 0:
            print(f"    assigned {e:,}/{len(D):,}")

    A["group_id"] = np.arange(len(A)); A["anchor_distance"] = 0
    D["group_id"] = gid;               D["anchor_distance"] = dist
    lost = D["group_id"] < 0
    if lost.any():
        D.loc[lost, "group_id"] = np.arange(10**8, 10**8 + int(lost.sum()))
        print(f"  WARNING: {int(lost.sum())} unhashable derivative(s) -> singleton groups")

    grouped = pd.concat([A, D], ignore_index=True)
    sizes = grouped.groupby("group_id").size()
    print(f"\nGroups: {len(sizes):,}   largest={sizes.max():,} "
          f"({100*sizes.max()/len(grouped):.3f}% of pool)   median={int(sizes.median())}")
    dd = D.loc[D.anchor_distance >= 0, "anchor_distance"]
    print(f"Anchor distance: median={int(dd.median())} p95={int(dd.quantile(.95))} "
          f"max={int(dd.max())} (of {NB} bits)")

    # ---- group-constrained split, stratified WITHIN each class ----
    rng = np.random.default_rng(FINAL_SEED)
    gsize = grouped.groupby("group_id").size().to_dict()
    gcls  = grouped.groupby("group_id")["class"].agg(lambda s: s.value_counts().idxmax()).to_dict()
    assign = {}
    for c in CLASS_NAMES:
        gids = [g for g, cc in gcls.items() if cc == c]
        rng.shuffle(gids)
        gids.sort(key=lambda g: -gsize[g])
        total = sum(gsize[g] for g in gids)
        placed = {k: 0 for k in TARGET_SPLIT}
        for g in gids:
            deficit = {k: TARGET_SPLIT[k] * total - placed[k] for k in TARGET_SPLIT}
            dest = max(deficit, key=deficit.get)
            assign[g] = dest; placed[dest] += gsize[g]
    grouped["split"] = grouped["group_id"].map(assign)

    # ---- leakage assertions ----
    print("\nPairwise overlap (must be zero):")
    okall = True
    for a_, b_ in [("train", "val"), ("train", "test"), ("val", "test")]:
        for col in ["filepath", "sha256", "group_id"]:
            ov = len(set(grouped[grouped.split == a_][col]) &
                     set(grouped[grouped.split == b_][col]))
            okall &= (ov == 0)
            print(f"  {a_}-{b_} {col:9s}: {ov}")
    assert okall, "LEAKAGE: a group spans partitions"
    print("PASS — zero cross-partition overlap.")

    ct = pd.crosstab(grouped["class"], grouped["split"])[["train", "val", "test"]]
    ratios = ct.div(ct.sum(1), axis=0)
    print("\nPer-class split ratios (target 0.70 / 0.15 / 0.15):")
    print(ratios.round(4).to_string())
    worst = float((ratios - np.array([0.70, 0.15, 0.15])).abs().max().max())
    print(f"\nWorst per-class deviation: {worst:.4f}  "
          f"{'OK' if worst <= 0.05 else '*** TOO LARGE — DO NOT PROCEED ***'}")
    print("\nCounts:"); print(ct.to_string())

    grouped.to_csv(manifest_path, index=False)
    manifest = grouped
    print(f"\nManifest saved -> {manifest_path}")

print("\n", manifest["split"].value_counts().to_string())


In [ ]:
# ============================================================
# BLOCK 2R — MODEL BUILDERS (fixes applied to ALL arms equally)
# ============================================================
# Changes vs the original Block 2, each applied identically to every arm so the
# ablation stays fair:
#   1. Gate projections initialise at ZERO -> gate starts at a uniform
#      sigmoid(0)=0.5 (a constant rescale the head's BatchNorm absorbs) instead
#      of random per-channel/per-pixel noise over pretrained features.
#   2. Spatial kernel 3 (not 7) on the 7x7 map. A 7x7 kernel on a 7x7 input is a
#      global op, not spatial attention.
#   3. Optional MULTI-SCALE branch: attention also applied at 14x14 (pool3_pool),
#      where a 3x3 spatial kernel is actually local. Branching off an existing
#      tensor, so no fragile mid-graph surgery.
#   4. Parameter-matched control: same added capacity, no gating.

from tensorflow.keras import layers, Model, backend as K
from tensorflow.keras.applications import DenseNet121
import tensorflow as tf

ATTENTION_INIT   = "zeros"   # set to "he_normal" to reproduce the old behaviour
SPATIAL_KERNEL   = 3
CHANNEL_RATIO    = 8
MULTI_SCALE      = True      # attention at 14x14 AND 7x7
UNFREEZE_LAST_N  = 120       # was 30; attention needs adaptable features


def channel_attention(x, ratio=CHANNEL_RATIO, init=ATTENTION_INIT):
    c = x.shape[-1]
    d1 = layers.Dense(max(1, c // ratio), activation="relu",
                      kernel_initializer="he_normal", use_bias=True)
    d2 = layers.Dense(c, kernel_initializer=init, bias_initializer="zeros",
                      use_bias=True)
    avg = layers.Reshape((1, 1, c))(layers.GlobalAveragePooling2D()(x))
    mx  = layers.Reshape((1, 1, c))(layers.GlobalMaxPooling2D()(x))
    g = layers.Add()([d2(d1(avg)), d2(d1(mx))])
    g = layers.Activation("sigmoid")(g)
    return layers.Multiply()([x, g])


def spatial_attention(x, kernel_size=SPATIAL_KERNEL, init=ATTENTION_INIT):
    avg = layers.Lambda(lambda t: K.mean(t, axis=-1, keepdims=True))(x)
    mx  = layers.Lambda(lambda t: K.max(t, axis=-1, keepdims=True))(x)
    cat = layers.Concatenate(axis=-1)([avg, mx])
    g = layers.Conv2D(1, kernel_size, padding="same", activation="sigmoid",
                      kernel_initializer=init, use_bias=False)(cat)
    return layers.Multiply()([x, g])


def cbam_block(x, ratio=CHANNEL_RATIO, kernel_size=SPATIAL_KERNEL):
    return spatial_attention(channel_attention(x, ratio), kernel_size)


def param_matched_block(x, ratio=CHANNEL_RATIO):
    """Same parameter increment as CBAM, no gating. Identity at init."""
    c = x.shape[-1]
    y = layers.Conv2D(max(1, c // ratio), 1, activation="relu",
                      kernel_initializer="he_normal")(x)
    y = layers.Conv2D(c, 1, kernel_initializer="zeros",
                      bias_initializer="zeros")(y)
    return layers.Add()([x, y])


def _head(x):
    x = layers.Dense(256, activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.40)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.30)(x)
    return layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32")(x)


def _backbone(seed, unfreeze_last_n=UNFREEZE_LAST_N):
    tf.keras.utils.set_random_seed(seed)
    base = DenseNet121(weights="imagenet", include_top=False,
                       input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = True
    for layer in base.layers[:-unfreeze_last_n]:
        layer.trainable = False
    return base


def _build(attn, seed, name, multi_scale=MULTI_SCALE):
    """attn: 'none' | 'channel' | 'spatial' | 'cbam' | 'param_matched'"""
    base = _backbone(seed)

    def apply(t):
        if attn == "none":          return t
        if attn == "channel":       return channel_attention(t)
        if attn == "spatial":       return spatial_attention(t)
        if attn == "cbam":          return cbam_block(t)
        if attn == "param_matched": return param_matched_block(t)
        raise ValueError(attn)

    taps = [base.output]                                   # 7x7x1024
    if multi_scale:
        taps.insert(0, base.get_layer("pool3_pool").output)  # 14x14x512
    pooled = [layers.GlobalAveragePooling2D()(apply(t)) for t in taps]
    x = layers.Concatenate()(pooled) if len(pooled) > 1 else pooled[0]
    return Model(base.input, _head(x), name=name)


MODEL_BUILDERS = {
    "DenseNet121":               lambda seed=42: _build("none",          seed, "DenseNet121"),
    "Channel-DenseNet121":       lambda seed=42: _build("channel",       seed, "Channel-DenseNet121"),
    "Spatial-DenseNet121":       lambda seed=42: _build("spatial",       seed, "Spatial-DenseNet121"),
    "CBAM-DenseNet121":          lambda seed=42: _build("cbam",          seed, "CBAM-DenseNet121"),
    "ParamMatched-DenseNet121":  lambda seed=42: _build("param_matched", seed, "ParamMatched-DenseNet121"),
}
ALL_CUSTOM_OBJECTS = {}

print(f"config: init={ATTENTION_INIT}  k={SPATIAL_KERNEL}  multi_scale={MULTI_SCALE}  "
      f"unfreeze={UNFREEZE_LAST_N}")
print("\nParameter-count sanity check:")
for name, builder in MODEL_BUILDERS.items():
    m = builder(seed=0)
    tr = int(sum(K.count_params(w) for w in m.trainable_weights))
    print(f"  {name:26s}: {m.count_params():>11,} total  {tr:>10,} trainable")
    del m; K.clear_session(); gc.collect()


In [ ]:
# ============================================================
# BLOCK 3R — TWO-STAGE TRAINING (applied identically to ALL arms)
# ============================================================
# Stage 1: backbone frozen. Head + attention learn sensible gates before the
#          pretrained features start moving. Higher LR.
# Stage 2: unfreeze UNFREEZE_LAST_N, fine-tune at a low LR with cosine decay.
# Plus: label smoothing, longer budget (your runs never triggered early stopping,
#       so val_loss was still improving at epoch 40 -> under-trained).

from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

STAGE1_EPOCHS   = 8
STAGE2_EPOCHS   = 40
STAGE1_LR       = 1e-3     # head + attention only, nothing pretrained is moving
STAGE2_LR       = 3e-5     # gentle fine-tuning of the unfrozen backbone
LABEL_SMOOTHING = 0.05
ES_PATIENCE     = 10
USE_TTA         = True     # horizontal-flip TTA at evaluation, all arms


def _freeze_backbone(model, frozen: bool):
    """DenseNet layers are everything before the first head/attention layer."""
    for layer in model.layers:
        if layer.name.startswith(("conv", "pool", "relu", "bn")):
            layer.trainable = not frozen


def evaluate_generator_tta(model, gen, use_tta=USE_TTA):
    gen.reset()
    prob = model.predict(gen, verbose=0)
    if use_tta:
        gen.reset()
        flipped = []
        for i in range(len(gen)):
            xb, _ = gen[i]
            flipped.append(model.predict(xb[:, :, ::-1, :], verbose=0))
        prob = (prob + np.concatenate(flipped)[:len(prob)]) / 2.0
    y_true = gen.classes[:len(prob)]
    y_pred = prob.argmax(1)
    return {
        "accuracy":            accuracy_score(y_true, y_pred),
        "balanced_accuracy":   balanced_accuracy_score(y_true, y_pred),
        "precision_weighted":  precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "recall_weighted":     recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "f1_weighted":         f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "roc_auc_ovr":         roc_auc_score(y_true, prob, average="macro", multi_class="ovr"),
    }


CLASS_WEIGHTS = compute_phase2_class_weights(manifest)
print("Phase II class weights:", CLASS_WEIGHTS)

results_path = RESULTS_DIR / "multiseed_results_v2.csv"
results_df = pd.read_csv(results_path) if results_path.exists() else pd.DataFrame()
done = set(zip(results_df.get("model", []), results_df.get("seed", [])))

for model_name, builder in MODEL_BUILDERS.items():
    for seed in SEEDS:
        if (model_name, seed) in done:
            print(f"skip {model_name} seed={seed}"); continue

        print("=" * 80)
        print(f"{model_name} | seed {seed}")
        print("=" * 80)
        set_global_seed(seed)
        train_gen, val_gen = make_generators_from_manifest(manifest, seed)
        model = builder(seed=seed)
        loss = CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING)
        ckpt = RESULTS_DIR / f"{model_name}_seed{seed}_best.keras"
        t0 = time.time()

        # ---------- Stage 1: frozen backbone ----------
        _freeze_backbone(model, True)
        model.compile(optimizer=Adam(STAGE1_LR), loss=loss, metrics=["accuracy"])
        print(f"  stage 1 ({STAGE1_EPOCHS} ep, lr={STAGE1_LR}) trainable="
              f"{int(sum(K.count_params(w) for w in model.trainable_weights)):,}")
        model.fit(train_gen, validation_data=val_gen, epochs=STAGE1_EPOCHS,
                  class_weight=CLASS_WEIGHTS, verbose=2)

        # ---------- Stage 2: fine-tune ----------
        _freeze_backbone(model, False)
        for layer in model.layers[:-UNFREEZE_LAST_N]:
            layer.trainable = False
        steps = max(1, len(train_gen)) * STAGE2_EPOCHS
        sched = tf.keras.optimizers.schedules.CosineDecay(STAGE2_LR, steps, alpha=0.02)
        model.compile(optimizer=Adam(sched), loss=loss, metrics=["accuracy"])
        print(f"  stage 2 ({STAGE2_EPOCHS} ep, cosine from {STAGE2_LR}) trainable="
              f"{int(sum(K.count_params(w) for w in model.trainable_weights)):,}")
        hist = model.fit(
            train_gen, validation_data=val_gen, epochs=STAGE2_EPOCHS,
            class_weight=CLASS_WEIGHTS, verbose=2,
            callbacks=[EarlyStopping(monitor="val_loss", patience=ES_PATIENCE,
                                     restore_best_weights=True),
                       ModelCheckpoint(str(ckpt), monitor="val_loss",
                                       save_best_only=True, mode="min")])

        elapsed = time.time() - t0
        m = evaluate_generator_tta(model, val_gen)
        m.update(model=model_name, seed=seed, train_seconds=elapsed,
                 epochs_trained=STAGE1_EPOCHS + len(hist.history["loss"]))
        results_df = pd.concat([results_df, pd.DataFrame([m])], ignore_index=True)
        results_df.to_csv(results_path, index=False)
        print(f"  -> acc={m['accuracy']:.4f} f1={m['f1_weighted']:.4f} "
              f"auc={m['roc_auc_ovr']:.4f}  ({elapsed/60:.1f} min)")

        del model, train_gen, val_gen
        K.clear_session(); gc.collect()

print("\n" + "=" * 80)
print("MULTI-SEED RESULTS v2")
print("=" * 80)
print(results_df.groupby("model")[
    ["accuracy","balanced_accuracy","precision_weighted",
     "recall_weighted","f1_weighted","roc_auc_ovr"]].agg(["mean","std"]).round(4))


In [ ]:
# ============================================================
# BLOCK 3R — TWO-STAGE TRAINING (applied identically to ALL arms)
# ============================================================
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.utils.class_weight import compute_class_weight

STAGE1_EPOCHS   = 8
STAGE2_EPOCHS   = 40
STAGE1_LR       = 1e-3
STAGE2_LR       = 3e-5
LABEL_SMOOTHING = 0.05
ES_PATIENCE     = 10
USE_TTA         = True


# ---- restored: these were in your ORIGINAL Block 3 and were lost when Block 3
# ---- got replaced wholesale. Nothing else changed vs your original versions.
def make_generators_from_manifest(manifest: pd.DataFrame, seed: int):
    train_df = manifest[manifest["split"] == "train"].reset_index(drop=True)
    val_df   = manifest[manifest["split"] == "val"].reset_index(drop=True)

    train_datagen = ImageDataGenerator(
        preprocessing_function=densenet_preprocess,
        rotation_range=15, width_shift_range=0.10, height_shift_range=0.10,
        zoom_range=0.10, horizontal_flip=True, brightness_range=(0.90, 1.10),
        fill_mode="nearest",
    )
    eval_datagen = ImageDataGenerator(preprocessing_function=densenet_preprocess)

    train_gen = train_datagen.flow_from_dataframe(
        train_df, x_col="filepath", y_col="class", target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE, class_mode="categorical", classes=CLASS_NAMES,
        shuffle=True, seed=seed)
    val_gen = eval_datagen.flow_from_dataframe(
        val_df, x_col="filepath", y_col="class", target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE, class_mode="categorical", classes=CLASS_NAMES,
        shuffle=False)
    return train_gen, val_gen


def compute_phase2_class_weights(manifest: pd.DataFrame) -> dict:
    train_df = manifest[manifest["split"] == "train"]
    y = train_df["class"].map({c: i for i, c in enumerate(CLASS_NAMES)}).to_numpy()
    w = compute_class_weight(class_weight="balanced", classes=np.arange(NUM_CLASSES), y=y)
    w = w / w.mean()
    return {i: float(x) for i, x in enumerate(w)}
# ---- end restored block ----


def _freeze_backbone(model, frozen: bool):
    """DenseNet layers are everything before the first head/attention layer."""
    for layer in model.layers:
        if layer.name.startswith(("conv", "pool", "relu", "bn")):
            layer.trainable = not frozen


def evaluate_generator_tta(model, gen, use_tta=USE_TTA):
    gen.reset()
    prob = model.predict(gen, verbose=0)
    if use_tta:
        gen.reset()
        flipped = []
        for i in range(len(gen)):
            xb, _ = gen[i]
            flipped.append(model.predict(xb[:, :, ::-1, :], verbose=0))
        prob = (prob + np.concatenate(flipped)[:len(prob)]) / 2.0
    y_true = gen.classes[:len(prob)]
    y_pred = prob.argmax(1)
    return {
        "accuracy":            accuracy_score(y_true, y_pred),
        "balanced_accuracy":   balanced_accuracy_score(y_true, y_pred),
        "precision_weighted":  precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "recall_weighted":     recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "f1_weighted":         f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "roc_auc_ovr":         roc_auc_score(y_true, prob, average="macro", multi_class="ovr"),
    }


CLASS_WEIGHTS = compute_phase2_class_weights(manifest)
print("Phase II class weights:", CLASS_WEIGHTS)

results_path = RESULTS_DIR / "multiseed_results_v2.csv"
results_df = pd.read_csv(results_path) if results_path.exists() else pd.DataFrame()
done = set(zip(results_df.get("model", []), results_df.get("seed", [])))

for model_name, builder in MODEL_BUILDERS.items():
    for seed in SEEDS:
        if (model_name, seed) in done:
            print(f"skip {model_name} seed={seed}"); continue

        print("=" * 80)
        print(f"{model_name} | seed {seed}")
        print("=" * 80)
        set_global_seed(seed)
        train_gen, val_gen = make_generators_from_manifest(manifest, seed)
        model = builder(seed=seed)
        loss = CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING)
        ckpt = RESULTS_DIR / f"{model_name}_seed{seed}_best.keras"
        t0 = time.time()

        _freeze_backbone(model, True)
        model.compile(optimizer=Adam(STAGE1_LR), loss=loss, metrics=["accuracy"])
        print(f"  stage 1 ({STAGE1_EPOCHS} ep, lr={STAGE1_LR}) trainable="
              f"{int(sum(K.count_params(w) for w in model.trainable_weights)):,}")
        model.fit(train_gen, validation_data=val_gen, epochs=STAGE1_EPOCHS,
                  class_weight=CLASS_WEIGHTS, verbose=2)

        _freeze_backbone(model, False)
        for layer in model.layers[:-UNFREEZE_LAST_N]:
            layer.trainable = False
        steps = max(1, len(train_gen)) * STAGE2_EPOCHS
        sched = tf.keras.optimizers.schedules.CosineDecay(STAGE2_LR, steps, alpha=0.02)
        model.compile(optimizer=Adam(sched), loss=loss, metrics=["accuracy"])
        print(f"  stage 2 ({STAGE2_EPOCHS} ep, cosine from {STAGE2_LR}) trainable="
              f"{int(sum(K.count_params(w) for w in model.trainable_weights)):,}")
        hist = model.fit(
            train_gen, validation_data=val_gen, epochs=STAGE2_EPOCHS,
            class_weight=CLASS_WEIGHTS, verbose=2,
            callbacks=[EarlyStopping(monitor="val_loss", patience=ES_PATIENCE,
                                     restore_best_weights=True),
                       ModelCheckpoint(str(ckpt), monitor="val_loss",
                                       save_best_only=True, mode="min")])

        elapsed = time.time() - t0
        m = evaluate_generator_tta(model, val_gen)
        m.update(model=model_name, seed=seed, train_seconds=elapsed,
                 epochs_trained=STAGE1_EPOCHS + len(hist.history["loss"]))
        results_df = pd.concat([results_df, pd.DataFrame([m])], ignore_index=True)
        results_df.to_csv(results_path, index=False)
        print(f"  -> acc={m['accuracy']:.4f} f1={m['f1_weighted']:.4f} "
              f"auc={m['roc_auc_ovr']:.4f}  ({elapsed/60:.1f} min)")

        del model, train_gen, val_gen
        K.clear_session(); gc.collect()

print("\n" + "=" * 80)
print("MULTI-SEED RESULTS v2")
print("=" * 80)
print(results_df.groupby("model")[
    ["accuracy","balanced_accuracy","precision_weighted",
     "recall_weighted","f1_weighted","roc_auc_ovr"]].agg(["mean","std"]).round(4))

In [ ]:
# ============================================================
# BLOCK 4 — STATISTICAL SIGNIFICANCE (Table 22)
# ============================================================
METRICS_FOR_TEST = ["accuracy", "precision_weighted", "recall_weighted",
                    "f1_weighted", "roc_auc_ovr"]
BASELINE_MODEL   = "DenseNet121"
COMPARISON_MODELS = ["Channel-DenseNet121", "Spatial-DenseNet121",
                     "CBAM-DenseNet121", "ParamMatched-DenseNet121"]


def mean_ci95(v):
    v = np.asarray(v, float); n = len(v); m = v.mean()
    if n < 2: return m, np.nan, np.nan
    h = stats.t.ppf(0.975, n - 1) * v.std(ddof=1) / np.sqrt(n)
    return m, m - h, m + h


def holm_bonferroni(p, alpha=0.05):
    p = np.asarray(p, float); m = len(p); order = np.argsort(p)
    adj = np.empty(m); run = 0.0
    for rank, idx in enumerate(order):
        run = max(run, (m - rank) * p[idx]); adj[idx] = min(run, 1.0)
    return adj, adj < alpha


base = results_df[results_df.model == BASELINE_MODEL].set_index("seed")
rows, raw_p = [], []
for cmp_model in COMPARISON_MODELS:
    var = results_df[results_df.model == cmp_model].set_index("seed")
    seeds = sorted(set(base.index) & set(var.index))
    if len(seeds) < 2:
        print(f"skip {cmp_model}: only {len(seeds)} paired seed(s)"); continue
    for metric in METRICS_FOR_TEST:
        a = base.loc[seeds, metric].to_numpy(float)
        b = var.loc[seeds, metric].to_numpy(float)
        d = b - a
        sh = stats.shapiro(d).pvalue if len(d) >= 3 and np.ptp(d) > 0 else np.nan
        try:    w = stats.wilcoxon(d).pvalue
        except Exception: w = 1.0
        t = stats.ttest_rel(b, a).pvalue if np.ptp(d) > 0 else 1.0
        chosen = t if (not np.isnan(sh) and sh > 0.05) else w
        mb, lb, hb = mean_ci95(a); mv, lv, hv = mean_ci95(b); md, ld, hd = mean_ci95(d)
        rows.append({"comparison": f"{cmp_model} vs {BASELINE_MODEL}", "metric": metric,
                     "n_seeds": len(seeds), "mean_base": mb, "ci_base": f"[{lb:.4f}, {hb:.4f}]",
                     "mean_variant": mv, "ci_variant": f"[{lv:.4f}, {hv:.4f}]",
                     "mean_diff": md, "ci_diff": f"[{ld:.4f}, {hd:.4f}]",
                     "shapiro_p": sh, "wilcoxon_p": w, "ttest_p": t,
                     "cohens_d": float(d.mean()/d.std(ddof=1)) if d.std(ddof=1) > 0 else np.nan})
        raw_p.append(chosen)

sig_df = pd.DataFrame(rows)
adj, rej = holm_bonferroni(raw_p)
sig_df["holm_adjusted_p"] = adj
sig_df["significant_at_0.05"] = rej
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
print("=" * 120)
print("Table 22 — Component-ablation significance vs DenseNet121 baseline")
print("=" * 120)
print(sig_df.round(5).to_string(index=False))
sig_df.to_csv(RESULTS_DIR / "statistical_significance.csv", index=False)
print(f"\nSaved -> {RESULTS_DIR/'statistical_significance.csv'}")
print("\nNOTE: the row that decides the paper's claim is "
      "'CBAM-DenseNet121 vs ParamMatched-DenseNet121' equivalent — attention may be "
      "credited only with the margin over the capacity-matched control, not over the "
      "plain baseline.")


In [ ]:
# ============================================================
# BLOCK 4B — CBAM vs ParamMatched, DIRECTLY (the decisive comparison)
# ============================================================
# Block 4 compares every arm against the plain DenseNet121 baseline. That answers
# "does CBAM beat the baseline" -- it does NOT answer "does attention beat the
# same parameter budget with no gating", which is the comparison that actually
# separates attention from capacity. This computes that comparison directly,
# reusing the identical statistical machinery from Block 4.

cbam = results_df[results_df.model == "CBAM-DenseNet121"].set_index("seed")
ctrl = results_df[results_df.model == "ParamMatched-DenseNet121"].set_index("seed")
seeds = sorted(set(cbam.index) & set(ctrl.index))
print(f"paired seeds available: {seeds}")

rows2, raw_p2 = [], []
for metric in METRICS_FOR_TEST:
    a = ctrl.loc[seeds, metric].to_numpy(float)   # control = "base" for this comparison
    b = cbam.loc[seeds, metric].to_numpy(float)   # CBAM    = "variant"
    d = b - a
    sh = stats.shapiro(d).pvalue if len(d) >= 3 and np.ptp(d) > 0 else np.nan
    try:    w = stats.wilcoxon(d).pvalue
    except Exception: w = 1.0
    t = stats.ttest_rel(b, a).pvalue if np.ptp(d) > 0 else 1.0
    chosen = t if (not np.isnan(sh) and sh > 0.05) else w
    mb, lb, hb = mean_ci95(a); mv, lv, hv = mean_ci95(b); md, ld, hd = mean_ci95(d)
    rows2.append({"comparison": "CBAM-DenseNet121 vs ParamMatched-DenseNet121",
                  "metric": metric, "n_seeds": len(seeds),
                  "mean_base": mb, "ci_base": f"[{lb:.4f}, {hb:.4f}]",
                  "mean_variant": mv, "ci_variant": f"[{lv:.4f}, {hv:.4f}]",
                  "mean_diff": md, "ci_diff": f"[{ld:.4f}, {hd:.4f}]",
                  "shapiro_p": sh, "wilcoxon_p": w, "ttest_p": t,
                  "cohens_d": float(d.mean()/d.std(ddof=1)) if d.std(ddof=1) > 0 else np.nan})
    raw_p2.append(chosen)

sig_df2 = pd.DataFrame(rows2)
adj2, rej2 = holm_bonferroni(raw_p2)
sig_df2["holm_adjusted_p"] = adj2
sig_df2["significant_at_0.05"] = rej2

print("\n" + "=" * 120)
print("CBAM-DenseNet121 vs ParamMatched-DenseNet121 (attention isolated from capacity)")
print("=" * 120)
print(sig_df2.round(5).to_string(index=False))

# append to the same CSV Block 4 wrote, rather than a separate file, so
# statistical_significance.csv is the single source for Table 22
combined = pd.concat([sig_df, sig_df2], ignore_index=True)
combined.to_csv(RESULTS_DIR / "statistical_significance.csv", index=False)
print(f"\nAppended -> {RESULTS_DIR/'statistical_significance.csv'} "
      f"({len(sig_df2)} new row(s), {len(combined)} total)")

any_sig = bool(sig_df2["significant_at_0.05"].any())
print(f"\nCBAM significantly beats the parameter-matched control on >=1 metric: {any_sig}")
if any_sig:
    print("  -> the paper can credit ATTENTION specifically, not just added capacity.")
else:
    print("  -> report this explicitly: CBAM's margin over the plain baseline (if any)")
    print("     is not distinguishable from what the extra parameters alone provide.")

In [ ]:
# ============================================================
# BLOCK 5 — LOCKED TEST EVALUATION (Tables 28-33) — RUNS ONCE
# ============================================================
test_result_path = RESULTS_DIR / "phase2_test_results.json"
test_df = manifest[manifest["split"] == "test"].reset_index(drop=True)

print("=" * 80); print("PHASE II LOCKED TEST SET"); print("=" * 80)
print(f"Test images: {len(test_df):,}")
print(test_df["class"].value_counts().reindex(CLASS_NAMES, fill_value=0).to_string())

test_gen = ImageDataGenerator(preprocessing_function=densenet_preprocess).flow_from_dataframe(
    test_df, x_col="filepath", y_col="class", target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode="categorical", classes=CLASS_NAMES, shuffle=False)
print("class mapping:", test_gen.class_indices)

if test_result_path.exists():
    print(f"\n{test_result_path} EXISTS — refusing to re-run (locked-test guarantee).")
    print("Delete it deliberately if you must re-run, and say so in the paper.")
    all_test_results = json.load(open(test_result_path))
    print(json.dumps(all_test_results, indent=2)[:1500])
else:
    all_test_results = {}
    for model_name in MODEL_BUILDERS:
        ckpt = RESULTS_DIR / f"{model_name}_seed{FINAL_SEED}_best.keras"
        if not ckpt.exists():
            print(f"MISSING checkpoint {ckpt} — run Block 3 for seed {FINAL_SEED}"); continue
        print("\n" + "=" * 80); print(f"LOCKED TEST — {model_name}"); print("=" * 80)
        model = tf.keras.models.load_model(ckpt, custom_objects=ALL_CUSTOM_OBJECTS)

        test_gen.reset(); prob = model.predict(test_gen, verbose=1)
        if USE_TTA:
            test_gen.reset(); flip = []
            for i in range(len(test_gen)):
                xb, _ = test_gen[i]
                flip.append(model.predict(xb[:, :, ::-1, :], verbose=0))
            prob = (prob + np.concatenate(flip)[:len(prob)]) / 2.0
        y_true = test_gen.classes[:len(prob)]; y_pred = prob.argmax(1)

        m = {"accuracy": float(accuracy_score(y_true, y_pred)),
             "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
             "precision_weighted": float(precision_score(y_true, y_pred, average="weighted", zero_division=0)),
             "recall_weighted": float(recall_score(y_true, y_pred, average="weighted", zero_division=0)),
             "f1_weighted": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
             "roc_auc_ovr": float(roc_auc_score(y_true, prob, average="macro", multi_class="ovr")),
             "final_seed": int(FINAL_SEED), "n_test_images": int(len(y_true)), "tta": bool(USE_TTA)}
        print(json.dumps(m, indent=2))
        print(classification_report(y_true, y_pred, target_names=CLASS_NAMES,
                                    digits=4, zero_division=0))

        cm = confusion_matrix(y_true, y_pred, labels=np.arange(NUM_CLASSES))
        rs = cm.sum(1, keepdims=True)
        cmn = np.divide(cm.astype(float), rs, out=np.zeros_like(cm, float), where=rs != 0)
        fig, ax = plt.subplots(1, 2, figsize=(14, 5))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES,
                    yticklabels=CLASS_NAMES, ax=ax[0]); ax[0].set_title(f"{model_name} — counts")
        sns.heatmap(cmn, annot=True, fmt=".2%", cmap="Blues", xticklabels=CLASS_NAMES,
                    yticklabels=CLASS_NAMES, ax=ax[1]); ax[1].set_title(f"{model_name} — row-normalized")
        for a in ax: a.set_xlabel("Predicted"); a.set_ylabel("Actual")
        plt.tight_layout(); plt.savefig(FIGURES_DIR / f"confusion_matrix_{model_name}.png",
                                        dpi=300, bbox_inches="tight"); plt.show(); plt.close(fig)
        pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(
            RESULTS_DIR / f"phase2_test_confusion_matrix_{model_name}.csv")

        yb = label_binarize(y_true, classes=np.arange(NUM_CLASSES))
        fig, a2 = plt.subplots(figsize=(7, 6)); per_class = {}
        for i, c in enumerate(CLASS_NAMES):
            fpr, tpr, _ = roc_curve(yb[:, i], prob[:, i]); ac = auc(fpr, tpr)
            per_class[c] = float(ac); a2.plot(fpr, tpr, label=f"{c} (AUC={ac:.4f})")
        a2.plot([0, 1], [0, 1], "k--", alpha=.5); a2.legend(loc="lower right")
        a2.set_xlabel("FPR"); a2.set_ylabel("TPR"); a2.set_title(f"{model_name} — OvR ROC")
        plt.tight_layout(); plt.savefig(FIGURES_DIR / f"roc_curves_{model_name}.png",
                                        dpi=300, bbox_inches="tight"); plt.show(); plt.close(fig)
        m["per_class_roc_auc"] = per_class
        np.savez_compressed(RESULTS_DIR / f"test_predictions_{model_name}.npz",
                            probs=prob, y_true=y_true)
        all_test_results[model_name] = m
        del model, prob; K.clear_session(); gc.collect()

    json.dump(all_test_results, open(test_result_path, "w"), indent=2)
    if all_test_results:
        cols = ["accuracy", "balanced_accuracy", "precision_weighted",
                "recall_weighted", "f1_weighted", "roc_auc_ovr"]
        tbl = pd.DataFrame(all_test_results).T[cols].sort_values("accuracy", ascending=False)
        print("\n" + "=" * 100); print("Table 28 — LOCKED TEST, ALL MODELS"); print("=" * 100)
        print(tbl.round(6).to_string())
        tbl.to_csv(RESULTS_DIR / "phase2_test_comparison_table.csv")


In [ ]:
# ============================================================
# BLOCK 6 — COMPUTATIONAL COMPLEXITY (Table 23)
# ============================================================
from tensorflow.python.framework.convert_to_constants import convert_variables_to_constants_v2


def get_flops(model, batch_size=1) -> int:
    shape = [batch_size] + list(model.inputs[0].shape[1:])
    cf = tf.function(lambda x: model(x)).get_concrete_function(
        tf.TensorSpec(shape, model.inputs[0].dtype))
    frozen = convert_variables_to_constants_v2(cf)
    with tf.Graph().as_default() as g:
        tf.graph_util.import_graph_def(frozen.graph.as_graph_def(), name="")
        opts = tf.compat.v1.profiler.ProfileOptionBuilder.float_operation()
        return tf.compat.v1.profiler.profile(
            graph=g, run_meta=tf.compat.v1.RunMetadata(), cmd="op", options=opts
        ).total_float_ops


def measure_latency(model, batch_size=1, warmup=10, runs=50) -> dict:
    x = np.random.rand(batch_size, IMG_SIZE, IMG_SIZE, 3).astype(np.float32)
    for _ in range(warmup): model.predict(x, verbose=0)
    ts = []
    for _ in range(runs):
        t0 = time.perf_counter(); model.predict(x, verbose=0)
        ts.append(time.perf_counter() - t0)
    ts = np.array(ts)
    return {"ms_per_image": float(np.median(ts) * 1000 / batch_size),
            "images_per_sec": float(batch_size / np.median(ts))}


rows, base_total = [], None
for name, builder in MODEL_BUILDERS.items():
    m = builder(seed=0)
    total = int(m.count_params())
    trainable = int(sum(K.count_params(w) for w in m.trainable_weights))
    base_total = base_total if base_total is not None else total
    tmp = RESULTS_DIR / f"_size_{name}.keras"; m.save(tmp)
    size_mb = tmp.stat().st_size / (1024 ** 2); tmp.unlink()
    f = get_flops(m); l1 = measure_latency(m, 1); l32 = measure_latency(m, BATCH_SIZE, runs=20)
    rows.append({"model": name, "total_params": total, "trainable_params": trainable,
                 "nontrainable_params": total - trainable,
                 "trainable_fraction_pct": 100 * trainable / total,
                 "delta_params_vs_baseline": total - base_total,
                 "model_size_MB_on_disk": size_mb, "GFLOPs_per_image": f / 1e9,
                 "latency_ms_per_image_bs1": l1["ms_per_image"],
                 "throughput_images_per_sec_bs1": l1["images_per_sec"],
                 "throughput_images_per_sec_bs32": l32["images_per_sec"]})
    print(f"  {name:26s} {total:>11,} params  {f/1e9:6.3f} GFLOPs  "
          f"{l1['ms_per_image']:6.2f} ms")
    del m; K.clear_session(); gc.collect()

complexity_df = pd.DataFrame(rows)
print("\n" + "=" * 110)
print("Table 23 — Computational complexity, measured")
print("=" * 110)
print(complexity_df.round(4).to_string(index=False))
complexity_df.to_csv(RESULTS_DIR / "computational_complexity.csv", index=False)


In [ ]:
# ============================================================
# BLOCK 7 — GATE-INITIALISATION ABLATION (Table 24)
# ============================================================
# The row that tells you whether CBAM's earlier deficit was a real property or an
# initialisation artifact. Report BOTH rows in the paper either way.
ABL_SEEDS = SEEDS[:3]
abl_path = RESULTS_DIR / "ablation_attention_config.csv"
abl_df = pd.read_csv(abl_path) if abl_path.exists() else pd.DataFrame()
done = set(zip(abl_df.get("setting", []), abl_df.get("seed", [])))

for init_mode, label in [("zeros", "Zero-init gate (proposed)"),
                         ("he_normal", "he_normal gate")]:
    for seed in ABL_SEEDS:
        if (label, seed) in done:
            print(f"skip {label} seed={seed}"); continue
        print(f"\n[Gate initialisation] {label} | seed {seed}")
        globals()["ATTENTION_INIT"] = init_mode      # builders read this at call time
        set_global_seed(seed)
        tg, vg = make_generators_from_manifest(manifest, seed)
        model = MODEL_BUILDERS["CBAM-DenseNet121"](seed=seed)
        loss = CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING)
        _freeze_backbone(model, True)
        model.compile(optimizer=Adam(STAGE1_LR), loss=loss, metrics=["accuracy"])
        model.fit(tg, validation_data=vg, epochs=STAGE1_EPOCHS,
                  class_weight=CLASS_WEIGHTS, verbose=2)
        _freeze_backbone(model, False)
        for layer in model.layers[:-UNFREEZE_LAST_N]: layer.trainable = False
        sched = tf.keras.optimizers.schedules.CosineDecay(
            STAGE2_LR, max(1, len(tg)) * STAGE2_EPOCHS, alpha=0.02)
        model.compile(optimizer=Adam(sched), loss=loss, metrics=["accuracy"])
        model.fit(tg, validation_data=vg, epochs=STAGE2_EPOCHS,
                  class_weight=CLASS_WEIGHTS, verbose=2,
                  callbacks=[EarlyStopping(monitor="val_loss", patience=ES_PATIENCE,
                                           restore_best_weights=True)])
        m = evaluate_generator_tta(model, vg)
        m.update(factor="Gate initialisation", setting=label, seed=seed)
        abl_df = pd.concat([abl_df, pd.DataFrame([m])], ignore_index=True)
        abl_df.to_csv(abl_path, index=False)
        print(f"   -> acc={m['accuracy']:.4f}  f1={m['f1_weighted']:.4f}")
        del model, tg, vg; K.clear_session(); gc.collect()

globals()["ATTENTION_INIT"] = "zeros"   # restore
print("\n" + "=" * 80); print("Table 24 — Gate initialisation"); print("=" * 80)
print(abl_df.groupby("setting")[["accuracy", "f1_weighted", "roc_auc_ovr"]]
      .agg(["mean", "std"]).round(4).to_string())


In [ ]:
# ============================================================
# BLOCK 8 — EXPORT ALL PAPER TABLES
# ============================================================
out = []
out.append("TABLE 21 — Multi-seed validation (mean ± SD)")
agg = results_df.groupby("model")[["accuracy", "balanced_accuracy", "precision_weighted",
                                   "recall_weighted", "f1_weighted", "roc_auc_ovr"]].agg(["mean", "std"])
t21 = pd.DataFrame(index=agg.index)
for c in ["accuracy", "balanced_accuracy", "precision_weighted",
          "recall_weighted", "f1_weighted", "roc_auc_ovr"]:
    t21[c] = [f"{m:.4f} ± {s:.4f}" for m, s in zip(agg[(c, "mean")], agg[(c, "std")])]
out.append(t21.to_string())

for label, path in [("TABLE 22 — Significance", "statistical_significance.csv"),
                    ("TABLE 23 — Complexity", "computational_complexity.csv"),
                    ("TABLE 24 — Gate initialisation", "ablation_attention_config.csv")]:
    p = RESULTS_DIR / path
    out.append(f"\n{label}")
    out.append(pd.read_csv(p).round(5).to_string(index=False) if p.exists() else "  (not run)")

p = RESULTS_DIR / "phase2_test_comparison_table.csv"
out.append("\nTABLE 28 — Locked test")
out.append(pd.read_csv(p, index_col=0).round(5).to_string() if p.exists() else "  (not run)")

for i, model_name in enumerate(MODEL_BUILDERS, start=29):
    p = RESULTS_DIR / f"phase2_test_confusion_matrix_{model_name}.csv"
    out.append(f"\nTABLE {i} — Confusion matrix, {model_name}")
    out.append(pd.read_csv(p, index_col=0).to_string() if p.exists() else "  (not run)")

p = RESULTS_DIR / "phase2_test_results.json"
if p.exists():
    r = json.load(open(p))
    out.append("\nTABLE 33 — Per-class OvR AUC")
    out.append(pd.DataFrame({k: v.get("per_class_roc_auc", {})
                             for k, v in r.items()}).T.round(4).to_string())

text = "\n".join(out)
(RESULTS_DIR / "paper_tables.txt").write_text(text, encoding="utf-8")
print(text)
print(f"\n\nSaved -> {RESULTS_DIR/'paper_tables.txt'}")


In [ ]:
import sys
print(sys.executable)

In [ ]:
# ============================================================
# BLOCK 0 — IMPORTS, REPRODUCIBILITY, AND GLOBAL CONFIGURATION
# ============================================================
import os, gc, json, time, random, hashlib, re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from tensorflow.keras import layers, Model, backend as K
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input as densenet_preprocess
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report, roc_auc_score,
    roc_curve, auc,
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import label_binarize
from scipy import stats

# ---- pip installs that may not be present ----
import subprocess, sys
for pkg in ["imagehash"]:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
import imagehash
from PIL import Image

# ------------------------------------------------------------
# Global configuration
# ------------------------------------------------------------
CLASS_NAMES = ["NonDemented", "VeryMildDemented", "MildDemented", "ModerateDemented"]
NUM_CLASSES = len(CLASS_NAMES)
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 40                 # was 15 (Phase I setting); Phase II ablation gets a real budget + early stopping
N_SEEDS = 5
SEEDS = [13, 27, 42, 77, 101]
FINAL_SEED = 42             # pre-specified seed for the one-time locked Phase II test evaluation

TARGET_SPLIT = {"train": 0.70, "val": 0.15, "test": 0.15}

# Near-duplicate grouping: a wide (256-bit) average hash with an adaptive Hamming threshold.
# A narrow 64-bit hash collapses structural MRI slices into one mega-group (they share a lot of
# global layout -- black background, centered brain silhouette), so the threshold is auto-tightened
# until no single perceptual-similarity group dominates the pool.
NEAR_DUP_HASH_SIZE = 16
NEAR_DUP_HAMMING_THRESHOLD = 20      # starting point (~7.8% of 256 bits)
NEAR_DUP_MEGA_GROUP_FRAC = 0.05      # no single group may exceed 5% of the pool
NEAR_DUP_MIN_THRESHOLD = 2           # floor for the adaptive search

# DATA_ROOT: local folder holding the downloaded Kaggle datasets.
# Defaults to <your home folder>/Downloads; override via the DATA_ROOT env var if your
# datasets live somewhere else, e.g. `set DATA_ROOT=D:\data` before launching Jupyter on Windows.
DATA_ROOT = Path(os.environ.get("DATA_ROOT", str(Path.home() / "Downloads")))

RAW_DATASET_CANDIDATES = [
    DATA_ROOT / "rk kaggle mri v2",
]

WORK_DIR = Path("./working")
GOV_DIR = WORK_DIR / "phase2_governance"
GOV_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = WORK_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


def set_global_seed(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass


set_global_seed(42)


def find_raw_dataset_root() -> Path:
    for candidate in RAW_DATASET_CANDIDATES:
        root = Path(candidate)
        if root.exists():
            for sub in [root, root / "data"]:
                if sub.exists() and all((sub / split / c).exists()
                                         for split in ["train", "val"] for c in CLASS_NAMES):
                    return sub
    for p in DATA_ROOT.rglob("*"):
        if p.is_dir():
            children = {c.name for c in p.iterdir() if c.is_dir()}
            if set(CLASS_NAMES).issubset(children):
                return p.parent if p.name in ("train", "val") else p
    raise FileNotFoundError(
        "Could not locate the Augmented Alzheimer MRI Dataset V2. "
        f"Checked {RAW_DATASET_CANDIDATES} and searched recursively under {DATA_ROOT}. "
        "Confirm the extracted folder contains train/ and val/ subfolders with the four class names."
    )


RAW_ROOT = find_raw_dataset_root()
print("Raw dataset root:", RAW_ROOT)
print("TensorFlow:", tf.__version__)
print("GPUs visible:", tf.config.list_physical_devices("GPU"))
print(f"Config: EPOCHS={EPOCHS}, SEEDS={SEEDS}, FINAL_SEED={FINAL_SEED}")


## Block 1 — Phase II Dataset Governance

SHA-256 exact-duplicate removal, then **global** (not per-class) perceptual near-duplicate grouping,
then a group-constrained 70:15:15 split. Comparing hashes globally rather than only within each class
means the audit can actually detect a cross-class near-duplicate — a genuinely more serious problem
than a same-class one, since it would indicate either a labeling error or an augmentation pipeline
bug, and a per-class-only comparison is structurally blind to it. Any cross-class group found is
reported explicitly in the audit table below, not silently absorbed.


In [ ]:
# ============================================================
# BLOCK 1 — PHASE II DATASET GOVERNANCE (GLOBAL NEAR-DUPLICATE AUDIT)
# ============================================================

def gather_source_images() -> pd.DataFrame:
    rows = []
    for split_folder in ["train", "val"]:
        split_root = RAW_ROOT / split_folder
        if not split_root.exists():
            continue
        for class_name in CLASS_NAMES:
            class_dir = split_root / class_name
            if not class_dir.exists():
                continue
            for img_path in class_dir.rglob("*"):
                if img_path.is_file() and img_path.suffix.lower() in {
                    ".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"
                }:
                    rows.append({"filepath": str(img_path), "class": class_name})
    df = pd.DataFrame(rows, columns=["filepath", "class"])
    if df.empty:
        print(f"WARNING: no images found under {RAW_ROOT}. Directory listing:")
        for p in sorted(RAW_ROOT.rglob("*")):
            if p.is_dir():
                print(" ", p)
        raise RuntimeError(
            "gather_source_images found 0 images -- check RAW_ROOT and CLASS_NAMES "
            "against the directory listing printed above."
        )
    print(f"Consolidated pool: {len(df):,} images across {df['class'].nunique()} classes")
    print(df["class"].value_counts())
    return df


def sha256_of_file(path: str) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 16), b""):
            h.update(chunk)
    return h.hexdigest()


def exact_duplicate_audit(df: pd.DataFrame) -> pd.DataFrame:
    print("Hashing", len(df), "images with SHA-256 ...")
    df = df.copy()
    df["sha256"] = [sha256_of_file(p) for p in df["filepath"]]
    n_before = len(df)
    dup_group_sizes = df.groupby("sha256").size()
    n_dup_groups = int((dup_group_sizes > 1).sum())
    n_in_dup_groups = int(dup_group_sizes[dup_group_sizes > 1].sum())

    cross_class = df.groupby("sha256")["class"].nunique()
    n_cross_class = int((cross_class > 1).sum())

    df_dedup = df.drop_duplicates(subset="sha256", keep="first").reset_index(drop=True)
    n_after = len(df_dedup)

    print("=" * 70)
    print("Table 3-equivalent -- SHA-256 exact-duplicate audit")
    print("=" * 70)
    print(f"Input images                 : {n_before:,}")
    print(f"Unique SHA-256 hashes         : {df['sha256'].nunique():,}")
    print(f"Duplicate hash groups         : {n_dup_groups:,}")
    print(f"Images in duplicate groups    : {n_in_dup_groups:,}")
    print(f"Duplicate copies removed      : {n_before - n_after:,}")
    print(f"Cross-class duplicate groups  : {n_cross_class:,}")
    print(f"Deduplication rate R_dup      : {(n_before - n_after) / n_before * 100:.3f}%")
    if n_cross_class > 0:
        print(f"\nWARNING: {n_cross_class} exact-duplicate group(s) span more than one class label.")
        print("Inspect these before proceeding -- this indicates a labeling inconsistency, not just leakage.")
    return df_dedup


def average_hash_of_file(path: str, hash_size: int = NEAR_DUP_HASH_SIZE) -> imagehash.ImageHash:
    with Image.open(path) as im:
        return imagehash.average_hash(im.convert("L"), hash_size=hash_size)


class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.parent[rb] = ra


_POPCOUNT_LUT = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)


def _popcount_elementwise(arr: np.ndarray) -> np.ndarray:
    """Popcount of each uint64 element, shape-preserving."""
    bytes_view = arr.astype(">u8").view(np.uint8).reshape(arr.shape + (8,))
    return _POPCOUNT_LUT[bytes_view].sum(axis=-1)


def _hash_to_words(h: imagehash.ImageHash, n_words: int) -> np.ndarray:
    """Pack an arbitrarily wide perceptual hash into n_words fixed uint64 words."""
    bits = int(str(h), 16)
    mask = (1 << 64) - 1
    words = np.empty(n_words, dtype=np.uint64)
    for w in range(n_words):
        shift = (n_words - 1 - w) * 64
        words[w] = np.uint64((bits >> shift) & mask)
    return words


def near_duplicate_grouping_global(df: pd.DataFrame,
                                    initial_threshold: int = NEAR_DUP_HAMMING_THRESHOLD,
                                    hash_size: int = NEAR_DUP_HASH_SIZE,
                                    chunk_size: int = 1000,
                                    mega_group_frac: float = NEAR_DUP_MEGA_GROUP_FRAC,
                                    min_threshold: int = NEAR_DUP_MIN_THRESHOLD) -> pd.DataFrame:
    """Global (cross-class) perceptual-hash near-duplicate grouping via average hash + Union-Find.
    Uses a wide (hash_size^2-bit) fingerprint and an adaptive Hamming threshold: hashes are computed
    once, then the threshold is halved and the grouping rebuilt (no re-hashing) until no single
    connected component dominates the pool -- this guards against transitive over-chaining, a known
    failure mode of coarse average hashes on images that share global layout (e.g. registered MRI
    slices with a consistent black background / centered brain silhouette)."""
    print(f"Computing {hash_size}x{hash_size} ({hash_size*hash_size}-bit) perceptual hashes for the full pool ...")
    df = df.copy().reset_index(drop=True)
    n = len(df)
    n_words = (hash_size * hash_size) // 64
    hashes = np.empty((n, n_words), dtype=np.uint64)
    for i, p in enumerate(df["filepath"]):
        hashes[i] = _hash_to_words(average_hash_of_file(p, hash_size), n_words)
        if i % 8000 == 0:
            print(f"  hashed {i:,}/{n:,}")

    def build_groups(threshold):
        uf = UnionFind(n)
        for start in range(0, n, chunk_size):
            end = min(start + chunk_size, n)
            block = hashes[start:end]
            for start2 in range(start, n, chunk_size):
                end2 = min(start2 + chunk_size, n)
                block2 = hashes[start2:end2]
                xor = np.bitwise_xor(block[:, None, :], block2[None, :, :])
                dist = _popcount_elementwise(xor).sum(axis=-1)
                near_i, near_j = np.where(dist <= threshold)
                for a, b in zip(near_i, near_j):
                    gi, gj = start + a, start2 + b
                    if gi < gj:
                        uf.union(gi, gj)
        local_root_to_group = {}
        next_group_id = 0
        group_ids = np.empty(n, dtype=np.int64)
        for i2 in range(n):
            root = uf.find(i2)
            if root not in local_root_to_group:
                local_root_to_group[root] = next_group_id
                next_group_id += 1
            group_ids[i2] = local_root_to_group[root]
        return group_ids

    threshold = initial_threshold
    while True:
        print(f"\nBuilding near-duplicate groups at Hamming threshold {threshold} "
              f"(of {hash_size*hash_size} bits) ...")
        group_ids = build_groups(threshold)
        sizes = pd.Series(group_ids).value_counts()
        largest_frac = sizes.iloc[0] / n
        print(f"  -> {sizes.shape[0]:,} groups; largest group = {sizes.iloc[0]:,} images "
              f"({largest_frac:.2%} of pool)")
        if largest_frac <= mega_group_frac:
            break
        if threshold <= min_threshold:
            raise RuntimeError(
                f"Even at the minimum threshold ({min_threshold}), the largest perceptual-similarity "
                f"group still covers {largest_frac:.1%} of the pool. Consider a larger hash_size or a "
                f"different near-duplicate method for this dataset."
            )
        threshold = max(min_threshold, threshold // 2)

    df["group_id"] = group_ids
    print(f"\nFinal near-duplicate Hamming threshold used: {threshold} bits (of {hash_size*hash_size})")

    n_groups = df["group_id"].nunique()
    group_class_counts = df.groupby("group_id")["class"].nunique()
    cross_class_groups = group_class_counts[group_class_counts > 1]

    print(f"\nTotal perceptual-similarity groups: {n_groups:,} (from {n:,} images)")
    print(f"Cross-class perceptual-similarity groups: {len(cross_class_groups):,}")
    if len(cross_class_groups) > 0:
        print("\nNOTE -- cross-class near-duplicate groups detected:")
        print(df[df["group_id"].isin(cross_class_groups.index)]
              .groupby("group_id")["class"].apply(lambda s: sorted(s.unique())).to_string())
    print(df.groupby("class").size())
    return df


def group_constrained_split(df: pd.DataFrame, n_seed_candidates: int = 250) -> pd.DataFrame:
    """Equations (3)-(4): assign whole groups to a single partition. Stratification uses each
    group's MAJORITY class (relevant only for the rare cross-class groups reported above); every
    image in the group still moves to the same partition regardless of its own label."""
    def majority_class(s):
        return s.value_counts().idxmax()

    groups = df.groupby("group_id").agg(
        class_=("class", majority_class), size=("class", "size")
    ).reset_index()

    best_seed, best_score, best_assignment = None, np.inf, None
    for seed in range(n_seed_candidates):
        rng = np.random.RandomState(seed)
        assignment = {}
        for class_name, class_groups in groups.groupby("class_"):
            class_groups = class_groups.sample(frac=1.0, random_state=rng)
            cum = class_groups["size"].cumsum()
            total = cum.iloc[-1]
            train_cut = TARGET_SPLIT["train"] * total
            val_cut = (TARGET_SPLIT["train"] + TARGET_SPLIT["val"]) * total
            for gid, running in zip(class_groups["group_id"], cum):
                if running <= train_cut:
                    assignment[gid] = "train"
                elif running <= val_cut:
                    assignment[gid] = "val"
                else:
                    assignment[gid] = "test"

        split_sizes = df["group_id"].map(assignment).value_counts()
        total_n = len(df)
        observed = {k: split_sizes.get(k, 0) / total_n for k in ["train", "val", "test"]}
        score = sum(abs(observed[k] - TARGET_SPLIT[k]) for k in TARGET_SPLIT)
        if score < best_score:
            best_score, best_seed, best_assignment = score, seed, assignment

    df = df.copy()
    df["split"] = df["group_id"].map(best_assignment)
    print(f"Selected seed {best_seed} (ratio deviation score {best_score:.5f})")

    split_table = df.groupby(["split", "class"]).size().unstack(fill_value=0)
    missing = [s for s in ["train", "val", "test"] if s not in split_table.index]
    if missing:
        raise RuntimeError(
            f"group_constrained_split produced no images in partition(s) {missing}. "
            "Check the near-duplicate output above before re-running."
        )
    print(split_table.loc[["train", "val", "test"]])
    return df


def integrity_check(df: pd.DataFrame) -> None:
    splits = {s: set(df.loc[df["split"] == s, "group_id"]) for s in ["train", "val", "test"]}
    pairs = [("train", "val"), ("train", "test"), ("val", "test")]
    print("Pairwise group-overlap check (must all be zero):")
    all_zero = True
    for a, b in pairs:
        overlap = len(splits[a] & splits[b])
        all_zero &= (overlap == 0)
        print(f"  {a}-{b} overlap: {overlap}")
    assert all_zero, "Leakage detected: a perceptual-similarity group spans multiple partitions."
    print("PASS -- zero cross-partition group overlap.")


# ---- Run the full Phase II governance pipeline ----
manifest_path = GOV_DIR / "phase2_manifest.csv"
if manifest_path.exists():
    print(f"Existing manifest found at {manifest_path}, loading it instead of re-running governance.")
    manifest = pd.read_csv(manifest_path)
else:
    pool = gather_source_images()
    deduped = exact_duplicate_audit(pool)
    grouped = near_duplicate_grouping_global(deduped)
    manifest = group_constrained_split(grouped)
    integrity_check(manifest)
    manifest[["filepath", "class", "split", "group_id"]].to_csv(manifest_path, index=False)
    print(f"\nManifest saved to {manifest_path}")

print(manifest["split"].value_counts())


In [ ]:
# ============================================================
# BLOCK 1 — PHASE II DATASET GOVERNANCE (GLOBAL NEAR-DUPLICATE AUDIT)
# ============================================================

def gather_source_images() -> pd.DataFrame:
    rows = []
    for split_folder in ["train", "val"]:
        split_root = RAW_ROOT / split_folder
        if not split_root.exists():
            continue
        for class_name in CLASS_NAMES:
            class_dir = split_root / class_name
            if not class_dir.exists():
                continue
            for img_path in class_dir.rglob("*"):
                if img_path.is_file() and img_path.suffix.lower() in {
                    ".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"
                }:
                    rows.append({"filepath": str(img_path), "class": class_name})
    df = pd.DataFrame(rows, columns=["filepath", "class"])
    if df.empty:
        print(f"WARNING: no images found under {RAW_ROOT}. Directory listing:")
        for p in sorted(RAW_ROOT.rglob("*")):
            if p.is_dir():
                print(" ", p)
        raise RuntimeError(
            "gather_source_images found 0 images -- check RAW_ROOT and CLASS_NAMES "
            "against the directory listing printed above."
        )
    print(f"Consolidated pool: {len(df):,} images across {df['class'].nunique()} classes")
    print(df["class"].value_counts())
    return df


def sha256_of_file(path: str) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 16), b""):
            h.update(chunk)
    return h.hexdigest()


def exact_duplicate_audit(df: pd.DataFrame) -> pd.DataFrame:
    print("Hashing", len(df), "images with SHA-256 ...")
    df = df.copy()
    df["sha256"] = [sha256_of_file(p) for p in df["filepath"]]
    n_before = len(df)
    dup_group_sizes = df.groupby("sha256").size()
    n_dup_groups = int((dup_group_sizes > 1).sum())
    n_in_dup_groups = int(dup_group_sizes[dup_group_sizes > 1].sum())

    cross_class = df.groupby("sha256")["class"].nunique()
    n_cross_class = int((cross_class > 1).sum())

    df_dedup = df.drop_duplicates(subset="sha256", keep="first").reset_index(drop=True)
    n_after = len(df_dedup)

    print("=" * 70)
    print("Table 3-equivalent -- SHA-256 exact-duplicate audit")
    print("=" * 70)
    print(f"Input images                 : {n_before:,}")
    print(f"Unique SHA-256 hashes         : {df['sha256'].nunique():,}")
    print(f"Duplicate hash groups         : {n_dup_groups:,}")
    print(f"Images in duplicate groups    : {n_in_dup_groups:,}")
    print(f"Duplicate copies removed      : {n_before - n_after:,}")
    print(f"Cross-class duplicate groups  : {n_cross_class:,}")
    print(f"Deduplication rate R_dup      : {(n_before - n_after) / n_before * 100:.3f}%")
    if n_cross_class > 0:
        print(f"\nWARNING: {n_cross_class} exact-duplicate group(s) span more than one class label.")
        print("Inspect these before proceeding -- this indicates a labeling inconsistency, not just leakage.")
    return df_dedup


def average_hash_of_file(path: str, hash_size: int = NEAR_DUP_HASH_SIZE) -> imagehash.ImageHash:
    with Image.open(path) as im:
        return imagehash.average_hash(im.convert("L"), hash_size=hash_size)


class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.parent[rb] = ra


_POPCOUNT_LUT = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)


def _popcount_elementwise(arr: np.ndarray) -> np.ndarray:
    """Popcount of each uint64 element, shape-preserving."""
    bytes_view = arr.astype(">u8").view(np.uint8).reshape(arr.shape + (8,))
    return _POPCOUNT_LUT[bytes_view].sum(axis=-1)


def _hash_to_words(h: imagehash.ImageHash, n_words: int) -> np.ndarray:
    """Pack an arbitrarily wide perceptual hash into n_words fixed uint64 words."""
    bits = int(str(h), 16)
    mask = (1 << 64) - 1
    words = np.empty(n_words, dtype=np.uint64)
    for w in range(n_words):
        shift = (n_words - 1 - w) * 64
        words[w] = np.uint64((bits >> shift) & mask)
    return words


def near_duplicate_grouping_global(df: pd.DataFrame,
                                    initial_threshold: int = NEAR_DUP_HAMMING_THRESHOLD,
                                    hash_size: int = NEAR_DUP_HASH_SIZE,
                                    chunk_size: int = 1000,
                                    mega_group_frac: float = NEAR_DUP_MEGA_GROUP_FRAC,
                                    min_threshold: int = NEAR_DUP_MIN_THRESHOLD) -> pd.DataFrame:
    """Global (cross-class) perceptual-hash near-duplicate grouping via average hash + Union-Find.
    Hashes are computed once, then the Hamming threshold is halved and the grouping rebuilt (no
    re-hashing) until either no single group dominates the pool, or the floor threshold is reached
    -- at the floor, a large remaining group is reported as a likely genuine consequence of this
    dataset's algorithmic augmentation (many images are rotated/flipped/rescaled copies of a much
    smaller set of source scans), not treated as an error."""
    print(f"Computing {hash_size}x{hash_size} ({hash_size*hash_size}-bit) perceptual hashes for the full pool ...")
    df = df.copy().reset_index(drop=True)
    n = len(df)
    n_words = (hash_size * hash_size) // 64
    hashes = np.empty((n, n_words), dtype=np.uint64)
    for i, p in enumerate(df["filepath"]):
        hashes[i] = _hash_to_words(average_hash_of_file(p, hash_size), n_words)
        if i % 8000 == 0:
            print(f"  hashed {i:,}/{n:,}")

    def build_groups(threshold):
        uf = UnionFind(n)
        for start in range(0, n, chunk_size):
            end = min(start + chunk_size, n)
            block = hashes[start:end]
            for start2 in range(start, n, chunk_size):
                end2 = min(start2 + chunk_size, n)
                block2 = hashes[start2:end2]
                xor = np.bitwise_xor(block[:, None, :], block2[None, :, :])
                dist = _popcount_elementwise(xor).sum(axis=-1)
                near_i, near_j = np.where(dist <= threshold)
                for a, b in zip(near_i, near_j):
                    gi, gj = start + a, start2 + b
                    if gi < gj:
                        uf.union(gi, gj)
        local_root_to_group = {}
        next_group_id = 0
        group_ids = np.empty(n, dtype=np.int64)
        for i2 in range(n):
            root = uf.find(i2)
            if root not in local_root_to_group:
                local_root_to_group[root] = next_group_id
                next_group_id += 1
            group_ids[i2] = local_root_to_group[root]
        return group_ids

    threshold = initial_threshold
    forced_floor = False
    while True:
        print(f"\nBuilding near-duplicate groups at Hamming threshold {threshold} "
              f"(of {hash_size*hash_size} bits) ...")
        group_ids = build_groups(threshold)
        sizes = pd.Series(group_ids).value_counts()
        largest_frac = sizes.iloc[0] / n
        print(f"  -> {sizes.shape[0]:,} groups; largest group = {sizes.iloc[0]:,} images "
              f"({largest_frac:.2%} of pool)")
        if largest_frac <= mega_group_frac:
            break
        if threshold <= min_threshold:
            forced_floor = True
            break
        threshold = max(min_threshold, threshold // 2)

    df["group_id"] = group_ids
    print(f"\nFinal near-duplicate Hamming threshold used: {threshold} bits (of {hash_size*hash_size})")

    if forced_floor:
        print("\n" + "!" * 80)
        print(f"NOTE: even at the minimum threshold ({min_threshold}), the largest perceptual-similarity")
        print(f"group still covers {largest_frac:.1%} of the pool ({sizes.iloc[0]:,} images). This dataset")
        print("is an AUGMENTED collection, so this most likely reflects a genuinely small number of unique")
        print("source scans that were heavily augmented (rotated/flipped/rescaled) to balance classes --")
        print("those augmented copies are legitimately near-duplicates of one another, so a large group is")
        print("an expected governance outcome here, not a bug. Proceeding with this grouping; check the")
        print("per-class split table below (and Table 8) to confirm partition sizes are still usable, and")
        print("report this concentration explicitly in the paper's Section 3.5.")
        print("!" * 80)
        print("\nLargest 10 group sizes:")
        print(sizes.head(10).to_string())

    n_groups = df["group_id"].nunique()
    group_class_counts = df.groupby("group_id")["class"].nunique()
    cross_class_groups = group_class_counts[group_class_counts > 1]

    print(f"\nTotal perceptual-similarity groups: {n_groups:,} (from {n:,} images)")
    print(f"Cross-class perceptual-similarity groups: {len(cross_class_groups):,}")
    if len(cross_class_groups) > 0:
        print("\nNOTE -- cross-class near-duplicate groups detected:")
        print(df[df["group_id"].isin(cross_class_groups.index)]
              .groupby("group_id")["class"].apply(lambda s: sorted(s.unique())).to_string())
    print(df.groupby("class").size())
    return df


def group_constrained_split(df: pd.DataFrame, n_seed_candidates: int = 250) -> pd.DataFrame:
    """Equations (3)-(4): assign whole groups to a single partition. Stratification uses each
    group's MAJORITY class (relevant only for the rare cross-class groups reported above); every
    image in the group still moves to the same partition regardless of its own label."""
    def majority_class(s):
        return s.value_counts().idxmax()

    groups = df.groupby("group_id").agg(
        class_=("class", majority_class), size=("class", "size")
    ).reset_index()

    best_seed, best_score, best_assignment = None, np.inf, None
    for seed in range(n_seed_candidates):
        rng = np.random.RandomState(seed)
        assignment = {}
        for class_name, class_groups in groups.groupby("class_"):
            class_groups = class_groups.sample(frac=1.0, random_state=rng)
            cum = class_groups["size"].cumsum()
            total = cum.iloc[-1]
            train_cut = TARGET_SPLIT["train"] * total
            val_cut = (TARGET_SPLIT["train"] + TARGET_SPLIT["val"]) * total
            for gid, running in zip(class_groups["group_id"], cum):
                if running <= train_cut:
                    assignment[gid] = "train"
                elif running <= val_cut:
                    assignment[gid] = "val"
                else:
                    assignment[gid] = "test"

        split_sizes = df["group_id"].map(assignment).value_counts()
        total_n = len(df)
        observed = {k: split_sizes.get(k, 0) / total_n for k in ["train", "val", "test"]}
        score = sum(abs(observed[k] - TARGET_SPLIT[k]) for k in TARGET_SPLIT)
        if score < best_score:
            best_score, best_seed, best_assignment = score, seed, assignment

    df = df.copy()
    df["split"] = df["group_id"].map(best_assignment)
    print(f"Selected seed {best_seed} (ratio deviation score {best_score:.5f})")

    split_table = df.groupby(["split", "class"]).size().unstack(fill_value=0)
    missing = [s for s in ["train", "val", "test"] if s not in split_table.index]
    if missing:
        raise RuntimeError(
            f"group_constrained_split produced no images in partition(s) {missing}. "
            "Check the near-duplicate output above before re-running."
        )
    print(split_table.loc[["train", "val", "test"]])
    return df


def integrity_check(df: pd.DataFrame) -> None:
    splits = {s: set(df.loc[df["split"] == s, "group_id"]) for s in ["train", "val", "test"]}
    pairs = [("train", "val"), ("train", "test"), ("val", "test")]
    print("Pairwise group-overlap check (must all be zero):")
    all_zero = True
    for a, b in pairs:
        overlap = len(splits[a] & splits[b])
        all_zero &= (overlap == 0)
        print(f"  {a}-{b} overlap: {overlap}")
    assert all_zero, "Leakage detected: a perceptual-similarity group spans multiple partitions."
    print("PASS -- zero cross-partition group overlap.")


# ---- Run the full Phase II governance pipeline ----
manifest_path = GOV_DIR / "phase2_manifest.csv"
if manifest_path.exists():
    print(f"Existing manifest found at {manifest_path}, loading it instead of re-running governance.")
    manifest = pd.read_csv(manifest_path)
else:
    pool = gather_source_images()
    deduped = exact_duplicate_audit(pool)
    grouped = near_duplicate_grouping_global(deduped)
    manifest = group_constrained_split(grouped)
    integrity_check(manifest)
    manifest[["filepath", "class", "split", "group_id"]].to_csv(manifest_path, index=False)
    print(f"\nManifest saved to {manifest_path}")

print(manifest["split"].value_counts())

## Block 2 — Matched Model Builders (4-Model Component Ablation)

Four architectures sharing an *identical* classification head and an *identical* frozen/unfrozen
backbone split. This isolates the individual contribution of channel attention and spatial
attention, not just their combined effect:

1. **DenseNet121** — no attention module (baseline).
2. **Channel-DenseNet121** — channel attention only.
3. **Spatial-DenseNet121** — spatial attention only.
4. **CBAM-DenseNet121** — sequential channel + spatial attention (the proposed model).


In [ ]:

# ============================================================
# BLOCK 2 — MODEL BUILDERS (4-MODEL ABLATION)
# ============================================================

def channel_attention(input_feature, ratio=8):
    channel = input_feature.shape[-1]
    shared_dense_one = layers.Dense(channel // ratio, activation="relu",
                                     kernel_initializer="he_normal", use_bias=True)
    shared_dense_two = layers.Dense(channel, kernel_initializer="he_normal", use_bias=True)

    avg_pool = layers.GlobalAveragePooling2D()(input_feature)
    avg_pool = layers.Reshape((1, 1, channel))(avg_pool)
    avg_pool = shared_dense_two(shared_dense_one(avg_pool))

    max_pool = layers.GlobalMaxPooling2D()(input_feature)
    max_pool = layers.Reshape((1, 1, channel))(max_pool)
    max_pool = shared_dense_two(shared_dense_one(max_pool))

    cbam_feature = layers.Add()([avg_pool, max_pool])
    cbam_feature = layers.Activation("sigmoid")(cbam_feature)
    return layers.Multiply()([input_feature, cbam_feature])


def spatial_attention(input_feature, kernel_size=7):
    avg_pool = layers.Lambda(lambda x: K.mean(x, axis=-1, keepdims=True))(input_feature)
    max_pool = layers.Lambda(lambda x: K.max(x, axis=-1, keepdims=True))(input_feature)
    concat = layers.Concatenate(axis=-1)([avg_pool, max_pool])
    cbam_feature = layers.Conv2D(filters=1, kernel_size=kernel_size, strides=1, padding="same",
                                  activation="sigmoid", kernel_initializer="he_normal",
                                  use_bias=False)(concat)
    return layers.Multiply()([input_feature, cbam_feature])


def cbam_block(input_feature, ratio=8, kernel_size=7):
    x = channel_attention(input_feature, ratio)
    x = spatial_attention(x, kernel_size)
    return x


def _classification_head(x):
    """Shared verbatim by all four models in the ablation."""
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.40)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.30)(x)
    return layers.Dense(NUM_CLASSES, activation="softmax")(x)


def _backbone(input_shape, unfreeze_last_n, seed):
    tf.keras.utils.set_random_seed(seed)
    base_model = DenseNet121(weights="imagenet", include_top=False, input_shape=input_shape)
    base_model.trainable = True
    for layer in base_model.layers[:-unfreeze_last_n]:
        layer.trainable = False
    return base_model


def build_densenet121(input_shape=(IMG_SIZE, IMG_SIZE, 3), unfreeze_last_n=30, seed=42):
    """Ablation baseline: DenseNet121 backbone + the shared head, no attention."""
    base_model = _backbone(input_shape, unfreeze_last_n, seed)
    outputs = _classification_head(base_model.output)
    return Model(inputs=base_model.input, outputs=outputs, name="DenseNet121")


def build_channel_densenet121(input_shape=(IMG_SIZE, IMG_SIZE, 3), unfreeze_last_n=30, seed=42):
    """Ablation arm: channel attention only."""
    base_model = _backbone(input_shape, unfreeze_last_n, seed)
    x = channel_attention(base_model.output, ratio=8)
    outputs = _classification_head(x)
    return Model(inputs=base_model.input, outputs=outputs, name="Channel-DenseNet121")


def build_spatial_densenet121(input_shape=(IMG_SIZE, IMG_SIZE, 3), unfreeze_last_n=30, seed=42):
    """Ablation arm: spatial attention only."""
    base_model = _backbone(input_shape, unfreeze_last_n, seed)
    x = spatial_attention(base_model.output, kernel_size=7)
    outputs = _classification_head(x)
    return Model(inputs=base_model.input, outputs=outputs, name="Spatial-DenseNet121")


def build_cbam_densenet121(input_shape=(IMG_SIZE, IMG_SIZE, 3), unfreeze_last_n=30, seed=42):
    """Proposed model: sequential channel + spatial attention (Section 4.3.2)."""
    base_model = _backbone(input_shape, unfreeze_last_n, seed)
    x = cbam_block(base_model.output, ratio=8, kernel_size=7)
    outputs = _classification_head(x)
    return Model(inputs=base_model.input, outputs=outputs, name="CBAM_DenseNet121")


MODEL_BUILDERS = {
    "DenseNet121": build_densenet121,
    "Channel-DenseNet121": build_channel_densenet121,
    "Spatial-DenseNet121": build_spatial_densenet121,
    "CBAM-DenseNet121": build_cbam_densenet121,
}
ALL_CUSTOM_OBJECTS = {
    "channel_attention": channel_attention,
    "spatial_attention": spatial_attention,
}

# Sanity check: confirm parameter counts increase in the expected direction
# (baseline < single-attention variants < combined CBAM).
print("Parameter-count sanity check:")
for name, builder in MODEL_BUILDERS.items():
    m = builder(seed=0)
    print(f"  {name:22s}: {m.count_params():,} total params")
    del m
    K.clear_session()
    gc.collect()


## Block 3 — Multi-Seed Training (4 Models × 5 Seeds)

Trains all **4 models** under **5 independent seeds each** (20 total runs) on the same Phase II
training/validation partitions. Checkpointing, early stopping, and LR scheduling all monitor
`val_loss` consistently (previously the checkpoint monitored `val_accuracy` while early stopping
and the LR schedule monitored `val_loss` — an inconsistency that could select a different epoch's
weights than the one early stopping was reasoning about). `EPOCHS = 40` with early stopping
(`patience=8`) means most runs will stop well before 40 epochs; 40 is a ceiling, not a target.

Resumable: re-running this cell skips any (model, seed) pair already present in
`results/multiseed_results.csv`.


In [ ]:

# ============================================================
# BLOCK 3 — MULTI-SEED TRAINING (4 MODELS x 5 SEEDS)
# ============================================================

def make_generators_from_manifest(manifest: pd.DataFrame, seed: int):
    train_df = manifest[manifest["split"] == "train"].reset_index(drop=True)
    val_df = manifest[manifest["split"] == "val"].reset_index(drop=True)

    train_datagen = ImageDataGenerator(
        preprocessing_function=densenet_preprocess,
        rotation_range=15, width_shift_range=0.10, height_shift_range=0.10,
        zoom_range=0.10, horizontal_flip=True, brightness_range=(0.90, 1.10),
        fill_mode="nearest",
    )
    eval_datagen = ImageDataGenerator(preprocessing_function=densenet_preprocess)

    train_gen = train_datagen.flow_from_dataframe(
        train_df, x_col="filepath", y_col="class", target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE, class_mode="categorical", classes=CLASS_NAMES,
        shuffle=True, seed=seed,
    )
    val_gen = eval_datagen.flow_from_dataframe(
        val_df, x_col="filepath", y_col="class", target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE, class_mode="categorical", classes=CLASS_NAMES,
        shuffle=False,
    )
    return train_gen, val_gen


def compute_phase2_class_weights(manifest: pd.DataFrame) -> dict:
    train_df = manifest[manifest["split"] == "train"]
    y = train_df["class"].map({c: i for i, c in enumerate(CLASS_NAMES)}).to_numpy()
    weights = compute_class_weight(class_weight="balanced", classes=np.arange(NUM_CLASSES), y=y)
    weights = weights / weights.mean()
    return {i: float(w) for i, w in enumerate(weights)}


def evaluate_generator(model, gen) -> dict:
    gen.reset()
    y_prob = model.predict(gen, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)
    y_true = gen.classes
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision_weighted": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "recall_weighted": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "roc_auc_ovr": roc_auc_score(y_true, y_prob, average="macro", multi_class="ovr"),
    }


CLASS_WEIGHTS = compute_phase2_class_weights(manifest)
print("Phase II class weights:", CLASS_WEIGHTS)

results_path = RESULTS_DIR / "multiseed_results.csv"
if results_path.exists():
    results_df = pd.read_csv(results_path)
else:
    results_df = pd.DataFrame(columns=["model", "seed", "accuracy", "balanced_accuracy",
                                        "precision_weighted", "recall_weighted", "f1_weighted",
                                        "roc_auc_ovr", "train_seconds", "epochs_trained"])

already_done = set(zip(results_df["model"], results_df["seed"]))

for model_name, builder in MODEL_BUILDERS.items():
    for seed in SEEDS:
        if (model_name, seed) in already_done:
            print(f"Skipping {model_name} seed={seed} (already in {results_path.name})")
            continue

        print("=" * 80)
        print(f"Training {model_name}  |  seed={seed}")
        print("=" * 80)
        set_global_seed(seed)

        train_gen, val_gen = make_generators_from_manifest(manifest, seed)
        model = builder(seed=seed)
        model.compile(optimizer=Adam(learning_rate=1e-4),
                       loss="categorical_crossentropy", metrics=["accuracy"])

        ckpt_path = RESULTS_DIR / f"{model_name}_seed{seed}_best.keras"
        callbacks = [
            EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
            ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-7),
            ModelCheckpoint(str(ckpt_path), monitor="val_loss", save_best_only=True, mode="min"),
        ]

        t0 = time.time()
        history = model.fit(train_gen, validation_data=val_gen, epochs=EPOCHS,
                             class_weight=CLASS_WEIGHTS, callbacks=callbacks, verbose=2)
        elapsed = time.time() - t0
        epochs_trained = len(history.history["loss"])

        metrics = evaluate_generator(model, val_gen)
        metrics.update({"model": model_name, "seed": seed, "train_seconds": elapsed,
                         "epochs_trained": epochs_trained})
        results_df = pd.concat([results_df, pd.DataFrame([metrics])], ignore_index=True)
        results_df.to_csv(results_path, index=False)
        print(f"Seed {seed} done in {elapsed/60:.1f} min ({epochs_trained} epochs). "
              f"Validation metrics: {metrics}")

        del model, train_gen, val_gen
        K.clear_session()
        gc.collect()

print("\n" + "=" * 80)
print("MULTI-SEED RESULTS (Phase II validation split, all 4 models)")
print("=" * 80)
summary = results_df.groupby("model")[
    ["accuracy", "balanced_accuracy", "precision_weighted", "recall_weighted", "f1_weighted", "roc_auc_ovr"]
].agg(["mean", "std"])
print(summary)


## Block 4 — Statistical Significance (with 95% CI and effect size)

Three seed-matched paired comparisons against the DenseNet121 baseline — Channel-only,
Spatial-only, and CBAM (combined) — across the 5 metrics recorded in Block 3 (15 tests total).
For each: Shapiro–Wilk tests the 5 paired differences for normality; both the Wilcoxon
signed-rank test and the paired t-test are reported; Holm–Bonferroni correction is applied across
all 15 tests jointly. A 95% confidence interval (t-distribution, n=5) is reported for each model's
mean per metric, and for the mean paired difference, alongside Cohen's d. With only 5 seeds, treat
the CIs as wide and the p-values as indicative — the CI width itself is informative about how much
weight the comparison can bear.


In [ ]:

# ============================================================
# BLOCK 4 — STATISTICAL SIGNIFICANCE (WITH 95% CI)
# ============================================================
METRICS_FOR_TEST = ["accuracy", "precision_weighted", "recall_weighted", "f1_weighted", "roc_auc_ovr"]
BASELINE_MODEL = "DenseNet121"
COMPARISON_MODELS = ["Channel-DenseNet121", "Spatial-DenseNet121", "CBAM-DenseNet121"]


def mean_ci95(values: np.ndarray) -> tuple:
    """95% CI for the mean via the t-distribution (appropriate for n=5)."""
    n = len(values)
    mean = values.mean()
    if n < 2:
        return mean, np.nan, np.nan
    sem = values.std(ddof=1) / np.sqrt(n)
    t_crit = stats.t.ppf(0.975, df=n - 1)
    return mean, mean - t_crit * sem, mean + t_crit * sem


def paired_cohens_d(diff: np.ndarray) -> float:
    return float(diff.mean() / diff.std(ddof=1)) if diff.std(ddof=1) > 0 else float("nan")


def holm_bonferroni(pvalues, alpha=0.05):
    order = np.argsort(pvalues)
    m = len(pvalues)
    adjusted = np.empty(m)
    reject = np.empty(m, dtype=bool)
    running_max = 0.0
    for rank, idx in enumerate(order):
        adj = (m - rank) * pvalues[idx]
        running_max = max(running_max, adj)
        adjusted[idx] = min(running_max, 1.0)
        reject[idx] = adjusted[idx] < alpha
    return adjusted, reject


pivot_base = results_df[results_df.model == BASELINE_MODEL].set_index("seed")[METRICS_FOR_TEST].sort_index()

rows = []
raw_pvalues = []
for comparison_model in COMPARISON_MODELS:
    pivot_cmp = results_df[results_df.model == comparison_model].set_index("seed")[METRICS_FOR_TEST].sort_index()
    common_seeds = sorted(set(pivot_base.index) & set(pivot_cmp.index))
    if len(common_seeds) < 2:
        print(f"Skipping {comparison_model}: fewer than 2 completed seeds paired with baseline.")
        continue
    a = pivot_base.loc[common_seeds]
    b = pivot_cmp.loc[common_seeds]

    for metric in METRICS_FOR_TEST:
        diff = (b[metric] - a[metric]).to_numpy()
        shapiro_p = stats.shapiro(diff).pvalue if len(diff) >= 3 else np.nan
        try:
            wilcoxon_p = stats.wilcoxon(diff).pvalue
        except ValueError:
            wilcoxon_p = np.nan
        ttest_p = stats.ttest_rel(b[metric], a[metric]).pvalue
        chosen_p = ttest_p if (not np.isnan(shapiro_p) and shapiro_p > 0.05) else wilcoxon_p

        mean_base, ci_lo_base, ci_hi_base = mean_ci95(a[metric].to_numpy())
        mean_cmp, ci_lo_cmp, ci_hi_cmp = mean_ci95(b[metric].to_numpy())
        mean_diff, ci_lo_diff, ci_hi_diff = mean_ci95(diff)

        rows.append({
            "comparison": f"{comparison_model} vs {BASELINE_MODEL}",
            "metric": metric,
            "n_seeds": len(common_seeds),
            f"mean_{BASELINE_MODEL}": mean_base,
            f"CI95_{BASELINE_MODEL}": f"[{ci_lo_base:.4f}, {ci_hi_base:.4f}]",
            f"mean_{comparison_model}": mean_cmp,
            f"CI95_{comparison_model}": f"[{ci_lo_cmp:.4f}, {ci_hi_cmp:.4f}]",
            "mean_diff": mean_diff,
            "CI95_diff": f"[{ci_lo_diff:.4f}, {ci_hi_diff:.4f}]",
            "shapiro_p_on_diff": shapiro_p,
            "wilcoxon_p": wilcoxon_p,
            "paired_ttest_p": ttest_p,
            "chosen_p_(normality-guided)": chosen_p,
            "cohens_d": paired_cohens_d(diff),
        })
        raw_pvalues.append(chosen_p)

sig_df = pd.DataFrame(rows)
adj_p, reject = holm_bonferroni(np.array(raw_pvalues))
sig_df["holm_adjusted_p"] = adj_p
sig_df["significant_at_0.05"] = reject

print("=" * 120)
print("Table: Component-ablation statistical significance (Channel-only, Spatial-only, CBAM vs. DenseNet121)")
print("=" * 120)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
print(sig_df.round(5).to_string(index=False))

sig_df.to_csv(RESULTS_DIR / "statistical_significance.csv", index=False)
print(f"\nSaved to {RESULTS_DIR / 'statistical_significance.csv'}")


## Block 5 — Phase II Locked Test Evaluation (pre-specified seed, all 4 models)

Runs **exactly once per model**. Each model's `FINAL_SEED = 42` checkpoint — pre-specified before
any test-set evaluation, not chosen after looking at validation results across seeds — is evaluated
on the untouched Phase II **test** partition. This avoids the subtle optimism of "pick whichever
of the 5 seeds did best on validation, then report its test score," which is itself a data-driven
selection step. The 5-seed runs in Block 3 exist to characterize *variability*; this block answers
"how does the pre-registered configuration perform," which are two different questions.

For every model this also generates a confusion matrix (raw + row-normalized) and one-vs-rest ROC
curves with per-class AUC — the visual ablation evidence, not just the scalar metrics table.

Guarded: will not re-run and overwrite `results/phase2_test_results.json` unless you delete it first.


In [ ]:

# ============================================================
# BLOCK 5 — PHASE II LOCKED TEST EVALUATION (ONE-TIME PER MODEL)
# ============================================================
test_result_path = RESULTS_DIR / "phase2_test_results.json"
FIGURES_DIR = RESULTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

test_df = manifest[manifest["split"] == "test"].reset_index(drop=True)
test_datagen = ImageDataGenerator(preprocessing_function=densenet_preprocess)
test_gen = test_datagen.flow_from_dataframe(
    test_df, x_col="filepath", y_col="class", target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode="categorical", classes=CLASS_NAMES, shuffle=False,
)


def plot_confusion_matrices(y_true, y_pred, model_name):
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(NUM_CLASSES))
    cm_norm = cm.astype(np.float64) / cm.sum(axis=1, keepdims=True)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES,
                yticklabels=CLASS_NAMES, ax=axes[0])
    axes[0].set_title(f"{model_name} — Confusion Matrix (counts)")
    axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Actual")

    sns.heatmap(cm_norm, annot=True, fmt=".2%", cmap="Blues", xticklabels=CLASS_NAMES,
                yticklabels=CLASS_NAMES, ax=axes[1])
    axes[1].set_title(f"{model_name} — Confusion Matrix (row-normalized)")
    axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Actual")

    plt.tight_layout()
    fig_path = FIGURES_DIR / f"confusion_matrix_{model_name}.png"
    plt.savefig(fig_path, dpi=150)
    plt.show()
    plt.close(fig)

    pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(
        RESULTS_DIR / f"phase2_test_confusion_matrix_{model_name}.csv")
    print(f"Saved {fig_path}")
    return cm


def plot_roc_curves(y_true, y_prob, model_name):
    y_true_bin = label_binarize(y_true, classes=np.arange(NUM_CLASSES))
    fig, ax = plt.subplots(figsize=(7, 6))
    per_class_auc = {}
    for i, class_name in enumerate(CLASS_NAMES):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_prob[:, i])
        roc_auc_i = auc(fpr, tpr)
        per_class_auc[class_name] = roc_auc_i
        ax.plot(fpr, tpr, label=f"{class_name} (AUC={roc_auc_i:.4f})")
    ax.plot([0, 1], [0, 1], "k--", alpha=0.5)
    ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
    ax.set_title(f"{model_name} — One-vs-Rest ROC Curves (Phase II test set)")
    ax.legend(loc="lower right")
    plt.tight_layout()
    fig_path = FIGURES_DIR / f"roc_curves_{model_name}.png"
    plt.savefig(fig_path, dpi=150)
    plt.show()
    plt.close(fig)
    print(f"Saved {fig_path}")
    return per_class_auc


if test_result_path.exists():
    print("Phase II test results already exist — refusing to re-run to preserve the")
    print("one-time-evaluation guarantee. Delete the file below to override:")
    print(test_result_path)
    print(json.dumps(json.load(open(test_result_path)), indent=2))
else:
    all_test_results = {}
    for model_name in MODEL_BUILDERS:
        ckpt_path = RESULTS_DIR / f"{model_name}_seed{FINAL_SEED}_best.keras"
        if not ckpt_path.exists():
            print(f"WARNING: checkpoint {ckpt_path} not found — run Block 3 for seed={FINAL_SEED} first. Skipping.")
            continue

        print("=" * 80)
        print(f"Phase II locked test evaluation — {model_name} (seed={FINAL_SEED})")
        print("=" * 80)
        model = tf.keras.models.load_model(ckpt_path, custom_objects=ALL_CUSTOM_OBJECTS, safe_mode=False)

        test_gen.reset()
        y_prob = model.predict(test_gen, verbose=0)
        y_pred = np.argmax(y_prob, axis=1)
        y_true = test_gen.classes

        metrics = {
            "accuracy": accuracy_score(y_true, y_pred),
            "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
            "precision_weighted": precision_score(y_true, y_pred, average="weighted", zero_division=0),
            "recall_weighted": recall_score(y_true, y_pred, average="weighted", zero_division=0),
            "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
            "roc_auc_ovr": roc_auc_score(y_true, y_prob, average="macro", multi_class="ovr"),
            "final_seed": FINAL_SEED,
            "checkpoint": str(ckpt_path),
            "n_test_images": len(test_df),
        }
        print(json.dumps(metrics, indent=2))
        print("\n", classification_report(y_true, y_pred, target_names=CLASS_NAMES))

        plot_confusion_matrices(y_true, y_pred, model_name)
        per_class_auc = plot_roc_curves(y_true, y_prob, model_name)
        metrics["per_class_roc_auc"] = per_class_auc

        all_test_results[model_name] = metrics

        del model
        K.clear_session()
        gc.collect()

    with open(test_result_path, "w") as f:
        json.dump(all_test_results, f, indent=2)

    print("\n" + "=" * 80)
    print("PHASE II LOCKED TEST RESULTS — all models, seed =", FINAL_SEED)
    print("=" * 80)
    comparison_table = pd.DataFrame(all_test_results).T[
        ["accuracy", "balanced_accuracy", "precision_weighted", "recall_weighted", "f1_weighted", "roc_auc_ovr"]
    ]
    print(comparison_table.round(4))
    comparison_table.to_csv(RESULTS_DIR / "phase2_test_comparison_table.csv")


In [ ]:
# ============================================================
# BLOCK 5 — PHASE II LOCKED TEST EVALUATION
# ONE-TIME EVALUATION PER MODEL
# ============================================================

import json
import gc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
)

from sklearn.preprocessing import label_binarize


# ============================================================
# 1. PATHS
# ============================================================

test_result_path = RESULTS_DIR / "phase2_test_results.json"

FIGURES_DIR = RESULTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# 2. PREPARE LOCKED TEST SET
# ============================================================

test_df = manifest[
    manifest["split"] == "test"
].reset_index(drop=True)

print("=" * 80)
print("PHASE II LOCKED TEST SET")
print("=" * 80)

print(f"Number of test images: {len(test_df)}")

print("\nClass distribution:")
print(
    test_df["class"]
    .value_counts()
    .reindex(CLASS_NAMES, fill_value=0)
)


test_datagen = ImageDataGenerator(
    preprocessing_function=densenet_preprocess
)

test_gen = test_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col="filepath",
    y_col="class",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    classes=CLASS_NAMES,
    shuffle=False
)

print("\nGenerator class mapping:")
print(test_gen.class_indices)


# ============================================================
# 3. CONFUSION MATRIX FUNCTION
# ============================================================

def plot_confusion_matrices(y_true, y_pred, model_name):

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=np.arange(NUM_CLASSES)
    )

    # Safe row normalization
    row_sums = cm.sum(axis=1, keepdims=True)

    cm_norm = np.divide(
        cm.astype(np.float64),
        row_sums,
        out=np.zeros_like(cm, dtype=np.float64),
        where=row_sums != 0
    )

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(14, 5)
    )

    # --------------------------------------------------------
    # Raw confusion matrix
    # --------------------------------------------------------

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES,
        ax=axes[0]
    )

    axes[0].set_title(
        f"{model_name} — Confusion Matrix (Counts)"
    )

    axes[0].set_xlabel("Predicted Class")
    axes[0].set_ylabel("Actual Class")


    # --------------------------------------------------------
    # Normalized confusion matrix
    # --------------------------------------------------------

    sns.heatmap(
        cm_norm,
        annot=True,
        fmt=".2%",
        cmap="Blues",
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES,
        ax=axes[1]
    )

    axes[1].set_title(
        f"{model_name} — Confusion Matrix (Row-Normalized)"
    )

    axes[1].set_xlabel("Predicted Class")
    axes[1].set_ylabel("Actual Class")


    plt.tight_layout()

    fig_path = (
        FIGURES_DIR /
        f"confusion_matrix_{model_name}.png"
    )

    plt.savefig(
        fig_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

    plt.close(fig)


    # --------------------------------------------------------
    # Save CSV
    # --------------------------------------------------------

    cm_df = pd.DataFrame(
        cm,
        index=CLASS_NAMES,
        columns=CLASS_NAMES
    )

    cm_csv_path = (
        RESULTS_DIR /
        f"phase2_test_confusion_matrix_{model_name}.csv"
    )

    cm_df.to_csv(cm_csv_path)


    cm_norm_df = pd.DataFrame(
        cm_norm,
        index=CLASS_NAMES,
        columns=CLASS_NAMES
    )

    cm_norm_csv_path = (
        RESULTS_DIR /
        f"phase2_test_confusion_matrix_normalized_{model_name}.csv"
    )

    cm_norm_df.to_csv(cm_norm_csv_path)


    print(f"\nSaved figure:")
    print(fig_path)

    print("\nSaved raw confusion matrix:")
    print(cm_csv_path)

    print("\nSaved normalized confusion matrix:")
    print(cm_norm_csv_path)

    return cm


# ============================================================
# 4. ROC CURVE FUNCTION
# ============================================================

def plot_roc_curves(
    y_true,
    y_prob,
    model_name
):

    # Convert integer labels to one-hot format
    y_true_bin = label_binarize(
        y_true,
        classes=np.arange(NUM_CLASSES)
    )

    fig, ax = plt.subplots(
        figsize=(7, 6)
    )

    per_class_auc = {}


    for i, class_name in enumerate(CLASS_NAMES):

        fpr, tpr, _ = roc_curve(
            y_true_bin[:, i],
            y_prob[:, i]
        )

        roc_auc_i = auc(
            fpr,
            tpr
        )

        per_class_auc[class_name] = float(
            roc_auc_i
        )

        ax.plot(
            fpr,
            tpr,
            label=f"{class_name} (AUC={roc_auc_i:.4f})"
        )


    # Random classifier reference
    ax.plot(
        [0, 1],
        [0, 1],
        "k--",
        alpha=0.5
    )


    ax.set_xlabel(
        "False Positive Rate"
    )

    ax.set_ylabel(
        "True Positive Rate"
    )

    ax.set_title(
        f"{model_name} — One-vs-Rest ROC Curves\n"
        "Phase II Locked Test Set"
    )

    ax.legend(
        loc="lower right"
    )

    plt.tight_layout()


    fig_path = (
        FIGURES_DIR /
        f"roc_curves_{model_name}.png"
    )

    plt.savefig(
        fig_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

    plt.close(fig)


    print("\nSaved ROC figure:")
    print(fig_path)

    return per_class_auc


# ============================================================
# 5. ONE-TIME LOCKED TEST EVALUATION
# ============================================================

if test_result_path.exists():

    print("=" * 80)

    print(
        "PHASE II TEST RESULTS ALREADY EXIST"
    )

    print("=" * 80)

    print(
        "Refusing to re-run evaluation to preserve "
        "the one-time locked-test guarantee."
    )

    print(
        "\nExisting results file:"
    )

    print(
        test_result_path
    )

    print("\nExisting results:\n")

    with open(
        test_result_path,
        "r"
    ) as f:

        existing_results = json.load(f)

    print(
        json.dumps(
            existing_results,
            indent=2
        )
    )


else:

    all_test_results = {}


    # ========================================================
    # EVALUATE EACH MODEL
    # ========================================================

    for model_name in MODEL_BUILDERS:


        # ----------------------------------------------------
        # Checkpoint path
        # ----------------------------------------------------

        ckpt_path = (
            RESULTS_DIR /
            f"{model_name}_seed{FINAL_SEED}_best.keras"
        )


        if not ckpt_path.exists():

            print("\nWARNING")

            print(
                f"Checkpoint not found:"
            )

            print(
                ckpt_path
            )

            print(
                f"\nRun Block 3 for seed={FINAL_SEED} first."
            )

            print(
                "Skipping this model."
            )

            continue


        # ----------------------------------------------------
        # Load model
        # ----------------------------------------------------

        print("\n")

        print("=" * 80)

        print(
            f"PHASE II LOCKED TEST EVALUATION"
        )

        print("=" * 80)

        print(
            f"Model      : {model_name}"
        )

        print(
            f"Final seed : {FINAL_SEED}"
        )

        print(
            f"Checkpoint : {ckpt_path}"
        )

        print(
            f"Test images: {len(test_df)}"
        )

        print("=" * 80)


        # IMPORTANT:
        # Do NOT use safe_mode=False
        # Your installed Keras version does not support it.

        try:

            model = tf.keras.models.load_model(
                ckpt_path,
                custom_objects=ALL_CUSTOM_OBJECTS
            )

        except Exception as e:

            print(
                f"\nERROR loading {model_name}"
            )

            print(e)

            print(
                "\nSkipping this model."
            )

            continue


        # ----------------------------------------------------
        # Reset generator
        # ----------------------------------------------------

        test_gen.reset()


        # ----------------------------------------------------
        # Prediction
        # ----------------------------------------------------

        print(
            "\nRunning predictions..."
        )

        y_prob = model.predict(
            test_gen,
            verbose=1
        )


        # ----------------------------------------------------
        # Predictions and ground truth
        # ----------------------------------------------------

        y_pred = np.argmax(
            y_prob,
            axis=1
        )

        y_true = test_gen.classes


        # ----------------------------------------------------
        # Safety check
        # ----------------------------------------------------

        assert len(y_true) == len(y_pred), (
            "Mismatch between true labels "
            "and predictions."
        )

        assert len(y_true) == len(test_df), (
            "Test sample count mismatch."
        )

        assert np.all(
            np.isfinite(y_prob)
        ), (
            "Non-finite probabilities detected."
        )


        # ====================================================
        # METRICS
        # ====================================================

        metrics = {

            "accuracy": float(
                accuracy_score(
                    y_true,
                    y_pred
                )
            ),

            "balanced_accuracy": float(
                balanced_accuracy_score(
                    y_true,
                    y_pred
                )
            ),

            "precision_weighted": float(
                precision_score(
                    y_true,
                    y_pred,
                    average="weighted",
                    zero_division=0
                )
            ),

            "recall_weighted": float(
                recall_score(
                    y_true,
                    y_pred,
                    average="weighted",
                    zero_division=0
                )
            ),

            "f1_weighted": float(
                f1_score(
                    y_true,
                    y_pred,
                    average="weighted",
                    zero_division=0
                )
            ),

            "roc_auc_ovr": float(
                roc_auc_score(
                    y_true,
                    y_prob,
                    average="macro",
                    multi_class="ovr"
                )
            ),

            "final_seed": int(
                FINAL_SEED
            ),

            "checkpoint": str(
                ckpt_path
            ),

            "n_test_images": int(
                len(test_df)
            )
        }


        # ====================================================
        # PRINT METRICS
        # ====================================================

        print("\n")

        print("=" * 80)

        print(
            f"TEST METRICS — {model_name}"
        )

        print("=" * 80)

        print(
            json.dumps(
                metrics,
                indent=2
            )
        )


        # ====================================================
        # CLASSIFICATION REPORT
        # ====================================================

        print("\n")

        print("=" * 80)

        print(
            "CLASSIFICATION REPORT"
        )

        print("=" * 80)

        print(
            classification_report(
                y_true,
                y_pred,
                target_names=CLASS_NAMES,
                digits=4,
                zero_division=0
            )
        )


        # ====================================================
        # CONFUSION MATRICES
        # ====================================================

        cm = plot_confusion_matrices(
            y_true,
            y_pred,
            model_name
        )


        # ====================================================
        # ROC CURVES
        # ====================================================

        per_class_auc = plot_roc_curves(
            y_true,
            y_prob,
            model_name
        )

        metrics[
            "per_class_roc_auc"
        ] = per_class_auc


        # ====================================================
        # STORE RESULTS
        # ====================================================

        all_test_results[
            model_name
        ] = metrics


        # ====================================================
        # CLEAN MEMORY
        # ====================================================

        del model
        del y_prob
        del y_pred

        K.clear_session()

        gc.collect()


    # ========================================================
    # SAVE LOCKED TEST RESULTS
    # ========================================================

    with open(
        test_result_path,
        "w"
    ) as f:

        json.dump(
            all_test_results,
            f,
            indent=2
        )


    # ========================================================
    # FINAL COMPARISON TABLE
    # ========================================================

    print("\n")

    print("=" * 100)

    print(
        f"PHASE II LOCKED TEST RESULTS — ALL MODELS"
    )

    print(
        f"FINAL SEED = {FINAL_SEED}"
    )

    print("=" * 100)


    if len(all_test_results) > 0:

        comparison_columns = [

            "accuracy",

            "balanced_accuracy",

            "precision_weighted",

            "recall_weighted",

            "f1_weighted",

            "roc_auc_ovr"

        ]


        comparison_table = pd.DataFrame(
            all_test_results
        ).T[
            comparison_columns
        ]


        # Sort by accuracy
        comparison_table = comparison_table.sort_values(
            by="accuracy",
            ascending=False
        )


        print(
            comparison_table.round(6)
        )


        comparison_csv_path = (
            RESULTS_DIR /
            "phase2_test_comparison_table.csv"
        )


        comparison_table.to_csv(
            comparison_csv_path
        )


        print("\n")

        print(
            "Comparison table saved:"
        )

        print(
            comparison_csv_path
        )


        print("\n")

        print(
            "Locked test results saved:"
        )

        print(
            test_result_path
        )


    else:

        print(
            "\nNo models were successfully evaluated."
        )


    print("\n")

    print("=" * 100)

    print(
        "PHASE II LOCKED TEST EVALUATION COMPLETE"
    )

    print("=" * 100)

## Block 6 — Computational Complexity and Inference Efficiency (4 models)

Measured parameter counts, on-disk model size, FLOPs, and inference latency/throughput for all
four ablation models, so the paper can report whether attention's accuracy gain (if any) comes at
a meaningful complexity cost.


In [ ]:

# ============================================================
# BLOCK 6 — COMPUTATIONAL COMPLEXITY AND INFERENCE EFFICIENCY
# ============================================================
from tensorflow.python.framework.convert_to_constants import convert_variables_to_constants_v2


def get_flops(keras_model, batch_size=1) -> int:
    input_shape = [batch_size] + list(keras_model.inputs[0].shape[1:])
    concrete_func = tf.function(lambda x: keras_model(x))
    concrete_func = concrete_func.get_concrete_function(
        tf.TensorSpec(input_shape, keras_model.inputs[0].dtype))
    frozen_func = convert_variables_to_constants_v2(concrete_func)
    graph_def = frozen_func.graph.as_graph_def()

    with tf.Graph().as_default() as graph:
        tf.graph_util.import_graph_def(graph_def, name="")
        run_meta = tf.compat.v1.RunMetadata()
        opts = tf.compat.v1.profiler.ProfileOptionBuilder.float_operation()
        flops = tf.compat.v1.profiler.profile(graph=graph, run_meta=run_meta, cmd="op", options=opts)
        return flops.total_float_ops


def measure_inference_latency(keras_model, batch_size=1, n_warmup=10, n_measure=100) -> dict:
    dummy = np.random.rand(batch_size, IMG_SIZE, IMG_SIZE, 3).astype(np.float32)
    for _ in range(n_warmup):
        keras_model.predict(dummy, verbose=0)
    times = []
    for _ in range(n_measure):
        t0 = time.perf_counter()
        keras_model.predict(dummy, verbose=0)
        times.append(time.perf_counter() - t0)
    times = np.array(times)
    return {
        "batch_size": batch_size,
        "mean_ms_per_image": float(times.mean() * 1000 / batch_size),
        "images_per_second": float(batch_size / times.mean()),
    }


complexity_rows = []
for model_name, builder in MODEL_BUILDERS.items():
    model = builder(seed=0)
    n_total = model.count_params()
    n_trainable = int(sum(K.count_params(w) for w in model.trainable_weights))
    n_nontrainable = n_total - n_trainable

    tmp_path = RESULTS_DIR / f"_{model_name}_sizecheck.keras"
    model.save(tmp_path)
    size_mb_on_disk = tmp_path.stat().st_size / (1024 ** 2)
    tmp_path.unlink()

    flops = get_flops(model, batch_size=1)
    lat_b1 = measure_inference_latency(model, batch_size=1)
    lat_b32 = measure_inference_latency(model, batch_size=BATCH_SIZE)

    complexity_rows.append({
        "model": model_name,
        "total_params": n_total,
        "trainable_params": n_trainable,
        "nontrainable_params": n_nontrainable,
        "trainable_fraction_pct": 100 * n_trainable / n_total,
        "model_size_MB_on_disk": size_mb_on_disk,
        "GFLOPs_per_image": flops / 1e9,
        "latency_ms_per_image_bs1": lat_b1["mean_ms_per_image"],
        "throughput_images_per_sec_bs1": lat_b1["images_per_second"],
        "throughput_images_per_sec_bs32": lat_b32["images_per_second"],
    })
    print(f"{model_name}: {n_total:,} params, {flops/1e9:.3f} GFLOPs, "
          f"{lat_b1['mean_ms_per_image']:.2f} ms/image (bs=1)")

    del model
    K.clear_session()
    gc.collect()

complexity_df = pd.DataFrame(complexity_rows)
print("\n" + "=" * 100)
print("Computational complexity — measured, all 4 models")
print("=" * 100)
print(complexity_df.round(4).to_string(index=False))
complexity_df.to_csv(RESULTS_DIR / "computational_complexity.csv", index=False)


## Block 7 — External Validation (subject-level, all 4 models)

**Dataset: `shreyanmohanty/oasis-alzheimers-detection-multi-class-dataset`** — a 4-class,
OASIS-1-derived dataset documenting 416 subjects with patient-level allocation. This replaces
`ninadaithal/imagesoasis` from v1, which does not document subject-level structure to the same
degree, making it harder to defend a "genuinely independent, subject-aware" evaluation claim.

**Subject-level aggregation.** A multi-slice-per-subject dataset evaluated at the slice level
overstates independence — many slices from the same subject can end up correctly or incorrectly
classified together, inflating apparent precision of the estimate. This block attempts to parse a
subject ID from each filename using common OASIS-style patterns (e.g. `OAS1_0001`), and if
successful, aggregates predictions per subject by **majority vote** before computing metrics. If no
consistent subject-ID pattern is found, it **falls back to slice-level evaluation and prints an
explicit warning** rather than silently claiming subject-level independence it cannot support —
inspect that warning before trusting the external-validation numbers.

If the dataset is not found under `DATA_ROOT`, this block raises a clear error naming the
exact dataset to download, rather than silently skipping external validation.


In [ ]:

# ============================================================
# BLOCK 7 — EXTERNAL VALIDATION (SUBJECT-LEVEL, OASIS-derived)
# ============================================================
EXTERNAL_DATASET_SLUG = "shreyanmohanty/oasis-alzheimers-detection-multi-class-dataset"
EXTERNAL_ROOT_CANDIDATES = [
    DATA_ROOT / "RK  OASIS MRI dataset" / "test",
    DATA_ROOT / "RK OASIS MRI dataset" / "test",
    DATA_ROOT / "RK  OASIS MRI dataset",
    DATA_ROOT / "RK OASIS MRI dataset",
    DATA_ROOT / "oasis-alzheimers-detection-multi-class-dataset",
]

# Common OASIS-style subject-ID patterns, tried in order. Add more here if your specific
# filenames don't match — the code below tells you explicitly if none matched.
SUBJECT_ID_PATTERNS = [
    re.compile(r"(OAS\d+[_-]?\d+)", re.IGNORECASE),      # e.g. OAS1_0001
    re.compile(r"(sub-?\d+)", re.IGNORECASE),             # e.g. sub-0001 / sub0001
    re.compile(r"^(\d+)[_-]"),                             # leading numeric token before _/-
]


def _normalize_label(name: str) -> str:
    return name.lower().replace(" ", "").replace("_", "").replace("-", "")


def find_external_root() -> Path:
    for candidate in EXTERNAL_ROOT_CANDIDATES:
        root = Path(candidate)
        if root.exists():
            children = [c for c in root.iterdir() if c.is_dir()]
            if children:
                return root
    for p in DATA_ROOT.rglob("*"):
        if p.is_dir():
            child_names = [c.name for c in p.iterdir() if c.is_dir()]
            normalized = {_normalize_label(n) for n in child_names}
            if len(normalized & {_normalize_label(c) for c in CLASS_NAMES}) >= 3:
                return p
    raise FileNotFoundError(
        f"External validation dataset not found. Download '{EXTERNAL_DATASET_SLUG}' from Kaggle "
        f"and place it under {DATA_ROOT}/oasis-alzheimers-detection-multi-class-dataset "
        f"(or set the DATA_ROOT env var), then re-run this cell."
    )


def build_label_mapping(external_root: Path) -> dict:
    external_folders = [c.name for c in external_root.iterdir() if c.is_dir()]
    norm_to_external = {_normalize_label(f): f for f in external_folders}
    norm_to_class = {_normalize_label(c): c for c in CLASS_NAMES}

    mapping = {}
    unmapped_classes = []
    for norm_class, class_name in norm_to_class.items():
        if norm_class in norm_to_external:
            mapping[norm_to_external[norm_class]] = class_name
        else:
            unmapped_classes.append(class_name)

    print("External folder -> harmonized class label:")
    for ext_folder, class_name in mapping.items():
        print(f"  {ext_folder!r:30s} -> {class_name}")
    if unmapped_classes:
        print(f"\nWARNING: could not confidently map these classes: {unmapped_classes}")
        print(f"Available external folders were: {external_folders}")
        print("Inspect the folder names above and extend `build_label_mapping` if needed.")
    return mapping


def extract_subject_id(filepath: str):
    filename = Path(filepath).name
    for pattern in SUBJECT_ID_PATTERNS:
        m = pattern.search(filename)
        if m:
            return m.group(1).upper().replace("-", "_")
    return None


external_root = find_external_root()
label_mapping = build_label_mapping(external_root)

if len(label_mapping) < NUM_CLASSES:
    raise ValueError(
        f"Only {len(label_mapping)}/{NUM_CLASSES} classes were mapped — resolve the label "
        f"mapping above before running external validation."
    )

ext_rows = []
for ext_folder, class_name in label_mapping.items():
    for img_path in (external_root / ext_folder).rglob("*"):
        if img_path.is_file() and img_path.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}:
            ext_rows.append({
                "filepath": str(img_path),
                "class": class_name,
                "subject_id": extract_subject_id(str(img_path)),
            })
external_manifest = pd.DataFrame(ext_rows)
print(f"\nExternal validation set: {len(external_manifest):,} images")
print(external_manifest["class"].value_counts())

n_parsed = external_manifest["subject_id"].notna().sum()
parse_rate = n_parsed / len(external_manifest) if len(external_manifest) else 0
n_unique_subjects = external_manifest["subject_id"].nunique()
print(f"\nSubject-ID parse rate: {parse_rate:.1%} ({n_parsed}/{len(external_manifest)} filenames matched)")
print(f"Unique subjects parsed: {n_unique_subjects:,}")

SUBJECT_LEVEL_OK = parse_rate > 0.95 and n_unique_subjects < len(external_manifest)
if not SUBJECT_LEVEL_OK:
    print("\n" + "!" * 80)
    print("WARNING: subject-ID parsing did not reliably succeed (either too many filenames failed")
    print("to match, or the parsed IDs are ~1-to-1 with images, i.e. no real grouping was found).")
    print("Falling back to SLICE-LEVEL (not subject-level) external evaluation.")
    print("If you know this dataset's exact filename convention, add a pattern to")
    print("SUBJECT_ID_PATTERNS above and re-run this cell before trusting subject-level claims.")
    print("!" * 80)
else:
    print(f"\nProceeding with SUBJECT-LEVEL aggregation across {n_unique_subjects:,} subjects.")


if not test_result_path.exists():
    raise RuntimeError("Run Block 5 first — external validation uses the same locked-seed checkpoints.")

ext_datagen = ImageDataGenerator(preprocessing_function=densenet_preprocess)
ext_gen = ext_datagen.flow_from_dataframe(
    external_manifest, x_col="filepath", y_col="class", target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode="categorical", classes=CLASS_NAMES, shuffle=False,
)

external_results = {}
for model_name in MODEL_BUILDERS:
    ckpt_path = RESULTS_DIR / f"{model_name}_seed{FINAL_SEED}_best.keras"
    if not ckpt_path.exists():
        print(f"WARNING: checkpoint {ckpt_path} not found. Skipping {model_name}.")
        continue

    print("=" * 80)
    print(f"External validation — {model_name} (seed={FINAL_SEED}, {EXTERNAL_DATASET_SLUG})")
    print("=" * 80)
    model = tf.keras.models.load_model(ckpt_path, custom_objects=ALL_CUSTOM_OBJECTS, safe_mode=False)

    ext_gen.reset()
    y_prob_slice = model.predict(ext_gen, verbose=0)
    y_pred_slice = np.argmax(y_prob_slice, axis=1)
    y_true_slice = ext_gen.classes

    eval_df = external_manifest.copy()
    eval_df["y_true"] = y_true_slice
    eval_df["y_pred_slice"] = y_pred_slice
    for i, c in enumerate(CLASS_NAMES):
        eval_df[f"prob_{c}"] = y_prob_slice[:, i]

    if SUBJECT_LEVEL_OK:
        # Aggregate by mean predicted probability per subject, then argmax (equivalent to
        # soft-voting across that subject's slices); majority vote on hard labels is reported too.
        subj_probs = eval_df.groupby("subject_id")[[f"prob_{c}" for c in CLASS_NAMES]].mean()
        y_pred_subject = subj_probs.values.argmax(axis=1)
        y_true_subject = eval_df.groupby("subject_id")["y_true"].agg(lambda s: s.value_counts().idxmax()).values
        y_true_eval, y_pred_eval, y_prob_eval = y_true_subject, y_pred_subject, subj_probs.values
        eval_unit = f"subject (n={len(subj_probs)})"
    else:
        y_true_eval, y_pred_eval, y_prob_eval = y_true_slice, y_pred_slice, y_prob_slice
        eval_unit = f"slice (n={len(eval_df)}) — subject-level parsing failed, see warning above"

    metrics = {
        "evaluation_unit": eval_unit,
        "accuracy": accuracy_score(y_true_eval, y_pred_eval),
        "balanced_accuracy": balanced_accuracy_score(y_true_eval, y_pred_eval),
        "precision_weighted": precision_score(y_true_eval, y_pred_eval, average="weighted", zero_division=0),
        "recall_weighted": recall_score(y_true_eval, y_pred_eval, average="weighted", zero_division=0),
        "f1_weighted": f1_score(y_true_eval, y_pred_eval, average="weighted", zero_division=0),
        "roc_auc_ovr": roc_auc_score(y_true_eval, y_prob_eval, average="macro", multi_class="ovr"),
        "dataset": EXTERNAL_DATASET_SLUG,
    }
    print(json.dumps(metrics, indent=2))
    external_results[model_name] = metrics

    del model
    K.clear_session()
    gc.collect()

with open(RESULTS_DIR / "external_validation_results.json", "w") as f:
    json.dump(external_results, f, indent=2)

internal_test_results = json.load(open(test_result_path))
print("\n" + "=" * 100)
print("Domain-shift comparison — internal Phase II locked test vs. external OASIS-derived set")
print("=" * 100)
for model_name in MODEL_BUILDERS:
    if model_name not in internal_test_results or model_name not in external_results:
        continue
    print(f"\n{model_name}:")
    for metric in METRICS_FOR_TEST:
        internal_v = internal_test_results[model_name][metric]
        external_v = external_results[model_name][metric]
        print(f"  {metric:20s}  internal={internal_v:.4f}   external={external_v:.4f}   "
              f"drop={internal_v - external_v:+.4f}")


In [ ]:
# ============================================================
# BLOCK 7 — EXTERNAL VALIDATION
# SUBJECT-LEVEL WHEN SUBJECT IDs CAN BE RELIABLY PARSED
# ============================================================

import re
import json
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras import backend as K
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)


# ============================================================
# 1. EXTERNAL DATASET CONFIGURATION
# ============================================================

EXTERNAL_DATASET_SLUG = (
    "shreyanmohanty/oasis-alzheimers-detection-multi-class-dataset"
)


# Possible dataset locations
EXTERNAL_ROOT_CANDIDATES = [

    DATA_ROOT / "RK  OASIS MRI dataset" / "test",
    DATA_ROOT / "RK OASIS MRI dataset" / "test",

    DATA_ROOT / "RK  OASIS MRI dataset",
    DATA_ROOT / "RK OASIS MRI dataset",

    DATA_ROOT / "oasis-alzheimers-detection-multi-class-dataset",

]


# Image extensions
IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".tif",
    ".tiff",
}


# ============================================================
# 2. SUBJECT-ID PATTERNS
# ============================================================

SUBJECT_ID_PATTERNS = [

    # Example:
    # OAS1_0001
    # OAS2_0001
    re.compile(
        r"(OAS\d+[_-]?\d+)",
        re.IGNORECASE
    ),

    # Example:
    # sub-0001
    # sub0001
    re.compile(
        r"(sub-?\d+)",
        re.IGNORECASE
    ),

    # Leading numeric token
    # Example:
    # 0001_image.jpg
    re.compile(
        r"^(\d+)[_-]"
    ),

]


# ============================================================
# 3. NORMALIZE CLASS NAMES
# ============================================================

def _normalize_label(name: str) -> str:

    return (
        str(name)
        .lower()
        .replace(" ", "")
        .replace("_", "")
        .replace("-", "")
    )


NORMALIZED_CLASS_NAMES = {
    _normalize_label(class_name): class_name
    for class_name in CLASS_NAMES
}


# ============================================================
# 4. CHECK WHETHER A DIRECTORY IS THE ACTUAL CLASS ROOT
# ============================================================

def is_class_root(path: Path) -> bool:

    if not path.exists() or not path.is_dir():
        return False

    child_dirs = [
        p
        for p in path.iterdir()
        if p.is_dir()
    ]

    normalized_children = {
        _normalize_label(p.name)
        for p in child_dirs
    }

    matched_classes = (
        normalized_children &
        set(NORMALIZED_CLASS_NAMES.keys())
    )

    return len(matched_classes) >= 3


# ============================================================
# 5. FIND ACTUAL EXTERNAL CLASS ROOT
# ============================================================

def find_external_root() -> Path:

    print("=" * 100)
    print("SEARCHING FOR EXTERNAL DATASET")
    print("=" * 100)


    # --------------------------------------------------------
    # First: check known candidate locations
    # --------------------------------------------------------

    for candidate in EXTERNAL_ROOT_CANDIDATES:

        candidate = Path(candidate)

        if not candidate.exists():
            continue


        print(f"\nChecking candidate:")

        print(candidate)


        # Candidate itself is class root
        if is_class_root(candidate):

            print(
                "FOUND CLASS ROOT:"
            )

            print(candidate)

            return candidate


        # Check immediate children
        for child in candidate.iterdir():

            if not child.is_dir():
                continue

            if is_class_root(child):

                print(
                    "FOUND NESTED CLASS ROOT:"
                )

                print(child)

                return child


        # Check recursively
        for root, dirs, files in __import__("os").walk(candidate):

            root_path = Path(root)

            if is_class_root(root_path):

                print(
                    "FOUND RECURSIVE CLASS ROOT:"
                )

                print(root_path)

                return root_path


    # --------------------------------------------------------
    # Second: recursively search DATA_ROOT
    # --------------------------------------------------------

    print(
        "\nSearching recursively under DATA_ROOT..."
    )

    for p in DATA_ROOT.rglob("*"):

        if not p.is_dir():
            continue

        if is_class_root(p):

            print(
                "FOUND CLASS ROOT:"
            )

            print(p)

            return p


    # --------------------------------------------------------
    # Nothing found
    # --------------------------------------------------------

    raise FileNotFoundError(

        "\nCould not locate the four-class external dataset.\n"

        f"Expected classes: {CLASS_NAMES}\n\n"

        "Please inspect the dataset directory structure."
    )


# ============================================================
# 6. BUILD LABEL MAPPING
# ============================================================

def build_label_mapping(
    external_root: Path
) -> dict:

    external_folders = [

        p.name
        for p in external_root.iterdir()
        if p.is_dir()

    ]


    print("\n")

    print("=" * 100)

    print(
        "EXTERNAL DATASET FOLDERS"
    )

    print("=" * 100)


    for folder in external_folders:

        print(
            folder
        )


    mapping = {}


    for folder in external_folders:

        normalized_folder = (
            _normalize_label(folder)
        )


        if normalized_folder in NORMALIZED_CLASS_NAMES:

            harmonized_class = (
                NORMALIZED_CLASS_NAMES[
                    normalized_folder
                ]
            )


            mapping[
                folder
            ] = harmonized_class


    print("\n")

    print("=" * 100)

    print(
        "EXTERNAL FOLDER -> HARMONIZED CLASS"
    )

    print("=" * 100)


    for external_folder, class_name in mapping.items():

        print(
            f"{external_folder:30s}"
            f" -> "
            f"{class_name}"
        )


    # --------------------------------------------------------
    # Check missing classes
    # --------------------------------------------------------

    mapped_classes = set(
        mapping.values()
    )


    missing_classes = [

        class_name
        for class_name in CLASS_NAMES
        if class_name not in mapped_classes

    ]


    if missing_classes:

        print("\nWARNING")

        print(
            "Could not map:"
        )

        print(
            missing_classes
        )


    return mapping


# ============================================================
# 7. EXTRACT SUBJECT ID
# ============================================================

def extract_subject_id(
    filepath: str
):

    filename = (
        Path(filepath)
        .name
    )


    for pattern in SUBJECT_ID_PATTERNS:

        match = pattern.search(
            filename
        )


        if match:

            return (
                match
                .group(1)
                .upper()
                .replace("-", "_")
            )


    return None


# ============================================================
# 8. FIND EXTERNAL ROOT
# ============================================================

external_root = (
    find_external_root()
)


print("\n")

print("=" * 100)

print(
    "FINAL EXTERNAL DATASET ROOT"
)

print("=" * 100)

print(
    external_root
)


# ============================================================
# 9. BUILD LABEL MAPPING
# ============================================================

label_mapping = (
    build_label_mapping(
        external_root
    )
)


if len(label_mapping) < NUM_CLASSES:

    raise ValueError(

        f"\nOnly {len(label_mapping)}/"
        f"{NUM_CLASSES} classes were mapped.\n"

        "The correct class folders could not be identified."
    )


# ============================================================
# 10. BUILD EXTERNAL MANIFEST
# ============================================================

ext_rows = []


for external_folder, class_name in label_mapping.items():

    class_folder = (
        external_root /
        external_folder
    )


    image_count = 0


    for img_path in class_folder.rglob("*"):

        if (
            img_path.is_file()
            and img_path.suffix.lower()
            in IMAGE_EXTENSIONS
        ):

            ext_rows.append({

                "filepath":
                    str(img_path),

                "class":
                    class_name,

                "subject_id":
                    extract_subject_id(
                        str(img_path)
                    ),

            })


            image_count += 1


    print(
        f"{class_name:25s}"
        f" : {image_count:,} images"
    )


external_manifest = pd.DataFrame(
    ext_rows
)


if len(external_manifest) == 0:

    raise RuntimeError(
        "No external images were found."
    )


print("\n")

print("=" * 100)

print(
    "EXTERNAL DATASET SUMMARY"
)

print("=" * 100)


print(
    f"Total images: "
    f"{len(external_manifest):,}"
)


print("\nClass distribution:")

print(
    external_manifest[
        "class"
    ]
    .value_counts()
    .reindex(
        CLASS_NAMES,
        fill_value=0
    )
)


# ============================================================
# 11. SUBJECT-ID PARSING AUDIT
# ============================================================

n_parsed = (
    external_manifest[
        "subject_id"
    ]
    .notna()
    .sum()
)


parse_rate = (

    n_parsed /
    len(external_manifest)

    if len(external_manifest) > 0

    else 0

)


n_unique_subjects = (

    external_manifest[
        "subject_id"
    ]
    .nunique()

)


print("\n")

print("=" * 100)

print(
    "SUBJECT-ID PARSING AUDIT"
)

print("=" * 100)


print(
    f"Parsed IDs: "
    f"{n_parsed:,} / "
    f"{len(external_manifest):,}"
)


print(
    f"Parse rate: "
    f"{parse_rate:.2%}"
)


print(
    f"Unique parsed subjects: "
    f"{n_unique_subjects:,}"
)


# ============================================================
# 12. DETERMINE EVALUATION LEVEL
# ============================================================

SUBJECT_LEVEL_OK = (

    parse_rate > 0.95

    and

    n_unique_subjects <
    len(external_manifest)

)


if SUBJECT_LEVEL_OK:

    print("\n")

    print("=" * 100)

    print(
        "SUBJECT-LEVEL EVALUATION ENABLED"
    )

    print("=" * 100)

    print(
        f"Aggregating predictions across "
        f"{n_unique_subjects:,} subjects."
    )


else:

    print("\n")

    print("!" * 100)

    print(
        "WARNING: SUBJECT-LEVEL PARSING FAILED"
    )

    print("!" * 100)

    print(
        "External validation will use "
        "SLICE-LEVEL evaluation."
    )

    print(
        "\nDo NOT claim this is subject-level "
        "external validation unless the filename "
        "pattern is corrected."
    )

    print("!" * 100)


# ============================================================
# 13. VERIFY INTERNAL TEST EXISTS
# ============================================================

if not test_result_path.exists():

    raise RuntimeError(

        "Run Block 5 first.\n"

        "External validation requires the "
        "same FINAL_SEED checkpoints used "
        "for the locked internal test."
    )


# ============================================================
# 14. CREATE EXTERNAL DATA GENERATOR
# ============================================================

ext_datagen = (
    ImageDataGenerator(
        preprocessing_function=
        densenet_preprocess
    )
)


ext_gen = (
    ext_datagen.flow_from_dataframe(

        dataframe=
            external_manifest,

        x_col=
            "filepath",

        y_col=
            "class",

        target_size=(
            IMG_SIZE,
            IMG_SIZE
        ),

        batch_size=
            BATCH_SIZE,

        class_mode=
            "categorical",

        classes=
            CLASS_NAMES,

        shuffle=
            False,

    )
)


print("\n")

print(
    "External generator class mapping:"
)

print(
    ext_gen.class_indices
)


# ============================================================
# 15. EXTERNAL VALIDATION
# ============================================================

external_results = {}


for model_name in MODEL_BUILDERS:


    # --------------------------------------------------------
    # Checkpoint
    # --------------------------------------------------------

    ckpt_path = (

        RESULTS_DIR /

        f"{model_name}_seed"
        f"{FINAL_SEED}_best.keras"

    )


    if not ckpt_path.exists():

        print("\nWARNING")

        print(
            f"Checkpoint not found:"
        )

        print(
            ckpt_path
        )

        print(
            f"Skipping {model_name}"
        )

        continue


    # --------------------------------------------------------
    # Load model
    # --------------------------------------------------------

    print("\n")

    print("=" * 100)

    print(
        f"EXTERNAL VALIDATION — "
        f"{model_name}"
    )

    print("=" * 100)


    print(
        f"Seed: "
        f"{FINAL_SEED}"
    )


    print(
        f"Dataset: "
        f"{EXTERNAL_DATASET_SLUG}"
    )


    print(
        f"Evaluation mode: "
        f"{'SUBJECT LEVEL' if SUBJECT_LEVEL_OK else 'SLICE LEVEL'}"
    )


    # IMPORTANT:
    # Do NOT use safe_mode=False
    # Older Keras versions do not support it.

    model = (
        tf.keras.models.load_model(

            ckpt_path,

            custom_objects=
                ALL_CUSTOM_OBJECTS

        )
    )


    # --------------------------------------------------------
    # Predict
    # --------------------------------------------------------

    ext_gen.reset()


    print(
        "\nRunning predictions..."
    )


    y_prob_slice = (
        model.predict(
            ext_gen,
            verbose=1
        )
    )


    y_pred_slice = (
        np.argmax(
            y_prob_slice,
            axis=1
        )
    )


    y_true_slice = (
        ext_gen.classes
    )


    # --------------------------------------------------------
    # Build evaluation dataframe
    # --------------------------------------------------------

    eval_df = (
        external_manifest
        .copy()
    )


    eval_df[
        "y_true"
    ] = y_true_slice


    eval_df[
        "y_pred_slice"
    ] = y_pred_slice


    for i, class_name in enumerate(
        CLASS_NAMES
    ):

        eval_df[
            f"prob_{class_name}"
        ] = y_prob_slice[:, i]


    # ========================================================
    # SUBJECT-LEVEL AGGREGATION
    # ========================================================

    if SUBJECT_LEVEL_OK:


        probability_columns = [

            f"prob_{class_name}"

            for class_name
            in CLASS_NAMES

        ]


        subj_probs = (

            eval_df

            .groupby(
                "subject_id"
            )

            [
                probability_columns
            ]

            .mean()

        )


        y_prob_eval = (
            subj_probs
            .values
        )


        y_pred_eval = (

            np.argmax(
                y_prob_eval,
                axis=1
            )

        )


        y_true_eval = (

            eval_df

            .groupby(
                "subject_id"
            )

            [
                "y_true"
            ]

            .agg(
                lambda x:
                x.value_counts()
                .idxmax()
            )

            .values

        )


        y_true_eval = (
            np.asarray(
                y_true_eval
            )
            .reshape(-1)
        )


        eval_unit = (

            f"subject-level "
            f"(n={len(subj_probs):,})"

        )


    # ========================================================
    # SLICE-LEVEL FALLBACK
    # ========================================================

    else:


        y_true_eval = (
            y_true_slice
        )


        y_pred_eval = (
            y_pred_slice
        )


        y_prob_eval = (
            y_prob_slice
        )


        eval_unit = (

            f"slice-level "
            f"(n={len(eval_df):,})"

        )


    # ========================================================
    # METRICS
    # ========================================================

    metrics = {


        "evaluation_unit":

            eval_unit,


        "accuracy":

            float(
                accuracy_score(
                    y_true_eval,
                    y_pred_eval
                )
            ),


        "balanced_accuracy":

            float(
                balanced_accuracy_score(
                    y_true_eval,
                    y_pred_eval
                )
            ),


        "precision_weighted":

            float(
                precision_score(
                    y_true_eval,
                    y_pred_eval,
                    average="weighted",
                    zero_division=0
                )
            ),


        "recall_weighted":

            float(
                recall_score(
                    y_true_eval,
                    y_pred_eval,
                    average="weighted",
                    zero_division=0
                )
            ),


        "f1_weighted":

            float(
                f1_score(
                    y_true_eval,
                    y_pred_eval,
                    average="weighted",
                    zero_division=0
                )
            ),


        "roc_auc_ovr":

            float(
                roc_auc_score(
                    y_true_eval,
                    y_prob_eval,
                    average="macro",
                    multi_class="ovr"
                )
            ),


        "dataset":

            EXTERNAL_DATASET_SLUG,


        "n_images":

            int(
                len(external_manifest)
            ),


        "n_evaluation_units":

            int(
                len(y_true_eval)
            ),


        "final_seed":

            int(
                FINAL_SEED
            ),

    }


    # ========================================================
    # PRINT RESULTS
    # ========================================================

    print("\n")

    print("=" * 100)

    print(
        f"EXTERNAL RESULTS — "
        f"{model_name}"
    )

    print("=" * 100)


    print(
        json.dumps(
            metrics,
            indent=2
        )
    )


    external_results[
        model_name
    ] = metrics


    # ========================================================
    # CLEAN MEMORY
    # ========================================================

    del model
    del y_prob_slice
    del y_pred_slice

    K.clear_session()

    gc.collect()


# ============================================================
# 16. SAVE EXTERNAL RESULTS
# ============================================================

external_results_path = (

    RESULTS_DIR /

    "external_validation_results.json"

)


with open(
    external_results_path,
    "w"
) as f:

    json.dump(
        external_results,
        f,
        indent=2
    )


print("\n")

print(
    "External validation results saved:"
)

print(
    external_results_path
)


# ============================================================
# 17. LOAD INTERNAL LOCKED TEST RESULTS
# ============================================================

with open(
    test_result_path,
    "r"
) as f:

    internal_test_results = (
        json.load(f)
    )


# ============================================================
# 18. INTERNAL VS EXTERNAL COMPARISON
# ============================================================

print("\n")

print("=" * 100)

print(
    "DOMAIN-SHIFT COMPARISON"
)

print(
    "INTERNAL LOCKED TEST vs EXTERNAL OASIS-DERIVED DATASET"
)

print("=" * 100)


comparison_rows = []


for model_name in MODEL_BUILDERS:


    if (

        model_name
        not in
        internal_test_results

        or

        model_name
        not in
        external_results

    ):

        continue


    print("\n")

    print(
        model_name
    )

    print(
        "-" * 80
    )


    row = {

        "model":

            model_name

    }


    for metric in METRICS_FOR_TEST:


        internal_value = (

            internal_test_results[
                model_name
            ][
                metric
            ]

        )


        external_value = (

            external_results[
                model_name
            ][
                metric
            ]

        )


        drop = (

            internal_value
            -
            external_value

        )


        row[
            f"internal_{metric}"
        ] = internal_value


        row[
            f"external_{metric}"
        ] = external_value


        row[
            f"drop_{metric}"
        ] = drop


        print(

            f"{metric:22s} "

            f"internal="
            f"{internal_value:.4f}   "

            f"external="
            f"{external_value:.4f}   "

            f"drop="
            f"{drop:+.4f}"

        )


    comparison_rows.append(
        row
    )


# ============================================================
# 19. SAVE DOMAIN-SHIFT COMPARISON
# ============================================================

domain_shift_df = (

    pd.DataFrame(
        comparison_rows
    )

)


domain_shift_path = (

    RESULTS_DIR /

    "phase2_internal_vs_external_comparison.csv"

)


domain_shift_df.to_csv(

    domain_shift_path,

    index=False

)


print("\n")

print("=" * 100)

print(
    "DOMAIN-SHIFT COMPARISON SAVED"
)

print("=" * 100)


print(
    domain_shift_path
)


print("\n")

print(
    "EXTERNAL VALIDATION COMPLETE"
)

print("=" * 100)

## Summary of Outputs

| File | Produced by | Content |
|---|---|---|
| `phase2_governance/phase2_manifest.csv` | Block 1 | Every image's SHA-256-deduplicated, globally-grouped, split-assigned manifest |
| `results/multiseed_results.csv` | Block 3 | Per-(model, seed) validation metrics — 20 rows once complete (4 models × 5 seeds) |
| `results/statistical_significance.csv` | Block 4 | 15 paired tests (3 comparisons × 5 metrics) with 95% CIs, Wilcoxon/paired-t, Holm–Bonferroni, Cohen's d |
| `results/phase2_test_results.json` | Block 5 | Locked, pre-specified-seed (42) test result for all 4 models |
| `results/phase2_test_comparison_table.csv` | Block 5 | The 4-model comparison table for the ablation section |
| `results/phase2_test_confusion_matrix_<model>.csv` + `figures/confusion_matrix_<model>.png` | Block 5 | Confusion matrices, all 4 models |
| `figures/roc_curves_<model>.png` | Block 5 | One-vs-rest ROC curves with per-class AUC, all 4 models |
| `results/computational_complexity.csv` | Block 6 | Measured params, FLOPs, latency, throughput, all 4 models |
| `results/external_validation_results.json` | Block 7 | Subject-level (or slice-level, with warning) OASIS-derived external performance, all 4 models |

These map directly onto the paper's Section 8 (component ablation, statistical significance,
confusion matrices, ROC curves), Section 8.4 (computational complexity), and Section 9 (locked
Phase II test + external validation).
